# 06. Walk-Forward Backtesting

## 1. Notebook Objective and Evaluation Framework

The previous modelling notebooks developed several approaches for forecasting Premier League home-win, draw and away-win probabilities:

1. tuned multinomial logistic regression;
2. tuned Random Forest;
3. tuned Histogram Gradient Boosting;
4. a validation-selected probability ensemble;
5. independent Poisson scoreline modelling.

These models were previously evaluated using one validation season and one locked test season.

That framework prevented the final test season from influencing model development. However, performance on a single unseen season may still be affected by season-specific conditions.

This notebook therefore introduces expanding-window walk-forward backtesting.

For each evaluation fold:

1. every season before the evaluation season forms the training sample;
2. preprocessing is fitted using the training data only;
3. the frozen model specifications are fitted from scratch;
4. the next season is predicted as completely unseen data;
5. the evaluation season is then added to the historical training window.

The intended evaluation structure is:

| Historical training period | Unseen season predicted |
|---|---|
| 2015–16 to 2019–20 | 2020–21 |
| 2015–16 to 2020–21 | 2021–22 |
| 2015–16 to 2021–22 | 2022–23 |
| 2015–16 to 2022–23 | 2023–24 |
| 2015–16 to 2023–24 | 2024–25 |

This produces five complete seasons of out-of-sample probability forecasts.

### Research Question

> Which model produces the strongest and most stable probability forecasts across several different unseen Premier League seasons?

### Why Walk-Forward Evaluation Matters

A model may perform especially well or poorly during one season because of:

- changing team quality;
- managerial changes;
- transfers;
- changes in scoring rates;
- changes in home advantage;
- unusual league conditions.

The 2020–21 season is particularly valuable because much of the season was played under exceptional attendance conditions.

Walk-forward evaluation therefore measures:

- average out-of-sample log loss;
- median out-of-sample log loss;
- variation between seasons;
- worst-season performance;
- the number of seasons won by each model;
- whether model rankings remain stable through time.

### Frozen Model Specifications

No new hyperparameter tuning will be performed in this notebook.

The previously selected specifications are frozen as:

- logistic regression: `C = 0.01`, `class_weight = None`;
- Random Forest: 500 trees, `max_depth = 5`, `min_samples_leaf = 30`, `max_features = 0.5`;
- Histogram Gradient Boosting: `learning_rate = 0.05`, `max_iter = 100`, `max_leaf_nodes = 3`, `min_samples_leaf = 30`, `l2_regularization = 1.0`;
- direct-model ensemble: 55% logistic regression, 15% Random Forest and 30% Histogram Gradient Boosting;
- Independent Poisson: home regularisation strength $\alpha_H = 0$ and away regularisation strength $\alpha_A = 0.001$.

Freezing these specifications prevents the walk-forward backtest from becoming another retrospective model-selection exercise.

## 2. Load and Validate the Walk-Forward Dataset

The processed modelling dataset contains the leakage-safe pre-match variables used throughout the classification notebooks.

The locally cached goal-target dataset created in Notebook 5 contains the observed full-time home and away goals required by the Independent Poisson models.

The two datasets will be merged using:

- season;
- fixture date;
- home team;
- away team.

The merge must preserve exactly one row per fixture and match all 3,800 Premier League matches.

The following columns are outcomes and must never enter the predictor matrix:

- full-time result;
- full-time home goals;
- full-time away goals.

This section will establish a single chronologically ordered dataframe for the walk-forward evaluation.

In [1]:
# ============================================================
# 2. Load and Validate the Walk-Forward Dataset
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Locate the project root
# ------------------------------------------------------------

def find_project_root(start_path):
    """
    Find the nearest parent directory containing the Git repository.
    """
    start_path = Path(start_path).resolve()

    for directory in [
        start_path,
        *start_path.parents,
    ]:
        if (directory / ".git").exists():
            return directory

    raise FileNotFoundError(
        "Could not locate the project root containing .git."
    )


def find_first_existing_column(
    dataframe,
    candidates,
    label,
):
    """
    Return the first candidate column present in a dataframe.
    """
    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate

    raise KeyError(
        f"Could not identify the {label} column. "
        f"Checked: {candidates}"
    )


project_root = find_project_root(Path.cwd())

processed_data_directory = (
    project_root
    / "data"
    / "processed"
)

raw_data_directory = (
    project_root
    / "data"
    / "raw"
)


# ------------------------------------------------------------
# Define the expected file locations
# ------------------------------------------------------------

processed_parquet_path = (
    processed_data_directory
    / "premier_league_model_data.parquet"
)

processed_csv_path = (
    processed_data_directory
    / "premier_league_model_data.csv"
)

goal_target_path = (
    raw_data_directory
    / "premier_league_goal_targets_2015_16_to_2024_25.csv"
)


# ------------------------------------------------------------
# Load the processed modelling dataset
# ------------------------------------------------------------

if processed_parquet_path.exists():

    model_data = pd.read_parquet(
        processed_parquet_path
    )

    modelling_file_used = (
        processed_parquet_path
    )

elif processed_csv_path.exists():

    model_data = pd.read_csv(
        processed_csv_path
    )

    modelling_file_used = (
        processed_csv_path
    )

else:

    raise FileNotFoundError(
        "Could not find the processed modelling dataset."
    )


# ------------------------------------------------------------
# Load the cached goal targets
# ------------------------------------------------------------

if not goal_target_path.exists():

    raise FileNotFoundError(
        "Could not find the cached goal-target dataset. "
        "Run Notebook 5 first."
    )


goal_targets = pd.read_csv(
    goal_target_path
)


# ------------------------------------------------------------
# Identify important modelling columns
# ------------------------------------------------------------

season_column = find_first_existing_column(
    model_data,
    candidates=[
        "Season",
        "season",
    ],
    label="season",
)

date_column = find_first_existing_column(
    model_data,
    candidates=[
        "Date",
        "date",
        "MatchDate",
        "match_date",
    ],
    label="fixture date",
)

home_team_column = find_first_existing_column(
    model_data,
    candidates=[
        "HomeTeam",
        "home_team",
        "Home",
    ],
    label="home-team",
)

away_team_column = find_first_existing_column(
    model_data,
    candidates=[
        "AwayTeam",
        "away_team",
        "Away",
    ],
    label="away-team",
)

target_column = find_first_existing_column(
    model_data,
    candidates=[
        "FTR",
        "Result",
        "result",
        "FullTimeResult",
    ],
    label="full-time result",
)


fixture_key_columns = [
    season_column,
    date_column,
    home_team_column,
    away_team_column,
]


# ------------------------------------------------------------
# Standardise modelling-dataset identifiers
# ------------------------------------------------------------

model_data = model_data.copy()

model_data[date_column] = pd.to_datetime(
    model_data[date_column],
    errors="raise",
    dayfirst=True,
).dt.normalize()

model_data[home_team_column] = (
    model_data[home_team_column]
    .astype(str)
    .str.strip()
)

model_data[away_team_column] = (
    model_data[away_team_column]
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# Standardise goal-target identifiers
# ------------------------------------------------------------

goal_targets = goal_targets.copy()

goal_targets["Date"] = pd.to_datetime(
    goal_targets["Date"],
    errors="raise",
).dt.normalize()

goal_targets["HomeTeam"] = (
    goal_targets["HomeTeam"]
    .astype(str)
    .str.strip()
)

goal_targets["AwayTeam"] = (
    goal_targets["AwayTeam"]
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# Retain and rename only the required goal columns
# ------------------------------------------------------------

goal_targets_for_merge = (
    goal_targets[
        [
            "Season",
            "Date",
            "HomeTeam",
            "AwayTeam",
            "FTHG",
            "FTAG",
        ]
    ]
    .rename(
        columns={
            "Season": season_column,
            "Date": date_column,
            "HomeTeam": home_team_column,
            "AwayTeam": away_team_column,
            "FTHG": "HomeGoalsTarget",
            "FTAG": "AwayGoalsTarget",
        }
    )
    .copy()
)


# ------------------------------------------------------------
# Validate uniqueness before merging
# ------------------------------------------------------------

assert not model_data.duplicated(
    subset=fixture_key_columns
).any(), (
    "Duplicate fixtures exist in the processed "
    "modelling dataset."
)

assert not goal_targets_for_merge.duplicated(
    subset=fixture_key_columns
).any(), (
    "Duplicate fixtures exist in the goal-target dataset."
)


# ------------------------------------------------------------
# Merge the goal targets onto the modelling dataset
# ------------------------------------------------------------

walk_forward_data = model_data.merge(
    goal_targets_for_merge,
    on=fixture_key_columns,
    how="left",
    validate="one_to_one",
    indicator=True,
)


# ------------------------------------------------------------
# Validate that every fixture matched
# ------------------------------------------------------------

unmatched_fixtures = (
    walk_forward_data.loc[
        walk_forward_data["_merge"]
        != "both",
        fixture_key_columns,
    ]
    .copy()
)

assert unmatched_fixtures.empty, (
    "At least one modelling fixture failed to match "
    "its home and away goal targets."
)

walk_forward_data = (
    walk_forward_data
    .drop(columns="_merge")
    .sort_values(
        by=[
            season_column,
            date_column,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validate the completed dataset
# ------------------------------------------------------------

assert len(walk_forward_data) == 3800, (
    "Expected 3,800 fixtures, but found "
    f"{len(walk_forward_data):,}."
)

assert (
    walk_forward_data[season_column]
    .nunique()
    ==
    10
), (
    "Expected ten Premier League seasons."
)

assert walk_forward_data[
    [
        target_column,
        "HomeGoalsTarget",
        "AwayGoalsTarget",
    ]
].notna().all().all(), (
    "At least one outcome target is missing."
)

assert set(
    walk_forward_data[
        target_column
    ].unique()
) == {
    "H",
    "D",
    "A",
}, (
    "The H/D/A target contains unexpected values."
)

assert (
    walk_forward_data[
        "HomeGoalsTarget"
    ]
    >= 0
).all(), (
    "Home-goal targets cannot be negative."
)

assert (
    walk_forward_data[
        "AwayGoalsTarget"
    ]
    >= 0
).all(), (
    "Away-goal targets cannot be negative."
)

assert np.allclose(
    walk_forward_data[
        "HomeGoalsTarget"
    ],
    walk_forward_data[
        "HomeGoalsTarget"
    ].round(),
), (
    "Home-goal targets must be whole numbers."
)

assert np.allclose(
    walk_forward_data[
        "AwayGoalsTarget"
    ],
    walk_forward_data[
        "AwayGoalsTarget"
    ].round(),
), (
    "Away-goal targets must be whole numbers."
)

walk_forward_data[
    "HomeGoalsTarget"
] = (
    walk_forward_data[
        "HomeGoalsTarget"
    ]
    .astype(int)
)

walk_forward_data[
    "AwayGoalsTarget"
] = (
    walk_forward_data[
        "AwayGoalsTarget"
    ]
    .astype(int)
)


# ------------------------------------------------------------
# Create a compact season summary
# ------------------------------------------------------------

walk_forward_season_summary = (
    walk_forward_data
    .groupby(
        season_column,
        sort=False,
    )
    .agg(
        Fixtures=(
            season_column,
            "size",
        ),
        MeanHomeGoals=(
            "HomeGoalsTarget",
            "mean",
        ),
        MeanAwayGoals=(
            "AwayGoalsTarget",
            "mean",
        ),
        FirstDate=(
            date_column,
            "min",
        ),
        LastDate=(
            date_column,
            "max",
        ),
    )
    .reset_index()
)

assert (
    walk_forward_season_summary[
        "Fixtures"
    ]
    ==
    380
).all(), (
    "Every season should contain 380 fixtures."
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print(
    "Walk-forward dataset loaded and validated successfully."
)

print(
    "Processed modelling file:",
    modelling_file_used.relative_to(
        project_root
    ),
)

print(
    "Goal-target file:",
    goal_target_path.relative_to(
        project_root
    ),
)

print(
    "Fixtures:",
    f"{len(walk_forward_data):,}",
)

print(
    "Seasons:",
    walk_forward_data[
        season_column
    ].nunique(),
)

display(
    walk_forward_season_summary.round(3)
)

display(
    walk_forward_data[
        fixture_key_columns
        + [
            target_column,
            "HomeGoalsTarget",
            "AwayGoalsTarget",
        ]
    ].head(10)
)

Walk-forward dataset loaded and validated successfully.
Processed modelling file: data\processed\premier_league_model_data.parquet
Goal-target file: data\raw\premier_league_goal_targets_2015_16_to_2024_25.csv
Fixtures: 3,800
Seasons: 10


C:\Users\kiera\AppData\Local\Temp\ipykernel_12216\1239936104.py:519: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  walk_forward_season_summary.round(3)


,Season,Fixtures,MeanHomeGoals,MeanAwayGoals,FirstDate,LastDate
0,2015-16,380,1.492,1.208,2015-08-08,2016-05-17
1,2016-17,380,1.597,1.203,2016-08-13,2017-05-21
2,2017-18,380,1.532,1.147,2017-08-11,2018-05-13
3,2018-19,380,1.568,1.253,2018-08-10,2019-05-12
4,2019-20,380,1.516,1.205,2019-08-09,2020-07-26
5,2020-21,380,1.353,1.342,2020-09-12,2021-05-23
6,2021-22,380,1.513,1.305,2021-08-13,2022-05-22
7,2022-23,380,1.634,1.218,2022-08-05,2023-05-28
8,2023-24,380,1.800,1.479,2023-08-11,2024-05-19
9,2024-25,380,1.513,1.421,2024-08-16,2025-05-25


,Season,Date,HomeTeam,AwayTeam,FTR,HomeGoalsTarget,AwayGoalsTarget
0,2015-16,2015-08-08,Bournemouth,Aston Villa,A,0,1
1,2015-16,2015-08-08,Chelsea,Swansea,D,2,2
2,2015-16,2015-08-08,Everton,Watford,D,2,2
3,2015-16,2015-08-08,Leicester,Sunderland,H,4,2
4,2015-16,2015-08-08,Man United,Tottenham,H,1,0
5,2015-16,2015-08-08,Norwich,Crystal Palace,A,1,3
6,2015-16,2015-08-09,Arsenal,West Ham,A,0,2
7,2015-16,2015-08-09,Newcastle,Southampton,D,2,2
8,2015-16,2015-08-09,Stoke,Liverpool,A,0,1
9,2015-16,2015-08-10,West Brom,Man City,A,0,3


## 3. Define Predictors and Walk-Forward Folds

The walk-forward backtest requires a fixed set of leakage-safe numeric predictors and a chronological sequence of expanding training windows.

The following columns must be excluded from the predictor matrix:

- season, date and team identifiers;
- full-time match result;
- full-time home goals;
- full-time away goals;
- any alternative post-match goal or result columns.

The remaining numeric columns form the modelling feature set.

The evaluation will cover the five most recent seasons:

- 2020–21;
- 2021–22;
- 2022–23;
- 2023–24;
- 2024–25.

For each evaluation season, every earlier season forms the training sample.

For example, the 2022–23 fold uses all fixtures from 2015–16 through 2021–22 for training and treats all 2022–23 fixtures as unseen data.

This section will validate that:

- no outcome information enters the predictor matrix;
- every evaluation season contains 380 fixtures;
- every training period ends before its evaluation season;
- the training window expands by one complete season at each fold.

In [2]:
# ============================================================
# 3. Define Predictors and Walk-Forward Folds
# ============================================================


# ------------------------------------------------------------
# Define outcome and identifier columns that cannot be predictors
# ------------------------------------------------------------

blocked_target_columns = {
    target_column,
    "HomeGoalsTarget",
    "AwayGoalsTarget",
    "FTHG",
    "FTAG",
    "FullTimeHomeGoals",
    "FullTimeAwayGoals",
    "HomeGoals",
    "AwayGoals",
    "Result",
    "FullTimeResult",
}

excluded_predictor_columns = (
    set(fixture_key_columns)
    |
    blocked_target_columns
)


# ------------------------------------------------------------
# Retain numeric pre-match predictors only
# ------------------------------------------------------------

feature_columns = [
    column
    for column in walk_forward_data.columns
    if (
        column not in excluded_predictor_columns
        and pd.api.types.is_numeric_dtype(
            walk_forward_data[column]
        )
    )
]


# ------------------------------------------------------------
# Validate the predictor set
# ------------------------------------------------------------

assert feature_columns, (
    "No numeric modelling predictors were identified."
)

blocked_columns_found = sorted(
    set(feature_columns)
    .intersection(
        blocked_target_columns
    )
)

assert not blocked_columns_found, (
    "Post-match outcome information entered the predictor set: "
    f"{blocked_columns_found}"
)

assert not set(
    fixture_key_columns
).intersection(
    feature_columns
), (
    "Fixture identifier columns entered the predictor set."
)

assert len(feature_columns) == len(
    set(feature_columns)
), (
    "Duplicate predictor names were detected."
)

assert walk_forward_data[
    feature_columns
].replace(
    [np.inf, -np.inf],
    np.nan,
).notna().any().all(), (
    "At least one predictor contains no usable finite values."
)


# ------------------------------------------------------------
# Identify the chronological season order
# ------------------------------------------------------------

ordered_seasons = (
    walk_forward_data[
        season_column
    ]
    .drop_duplicates()
    .tolist()
)

assert ordered_seasons == sorted(
    ordered_seasons
), (
    "The season labels are not in chronological order."
)

assert len(ordered_seasons) == 10, (
    "Expected ten seasons in chronological order."
)


# ------------------------------------------------------------
# Define the five walk-forward evaluation seasons
# ------------------------------------------------------------

evaluation_seasons = ordered_seasons[-5:]

expected_evaluation_seasons = [
    "2020-21",
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
]

assert evaluation_seasons == (
    expected_evaluation_seasons
), (
    "The identified evaluation seasons do not match "
    "the intended walk-forward design."
)


# ------------------------------------------------------------
# Construct the fold plan
# ------------------------------------------------------------

walk_forward_fold_records = []

for fold_number, evaluation_season in enumerate(
    evaluation_seasons,
    start=1,
):

    evaluation_position = (
        ordered_seasons.index(
            evaluation_season
        )
    )

    training_seasons = (
        ordered_seasons[
            :evaluation_position
        ]
    )

    training_mask = (
        walk_forward_data[
            season_column
        ]
        .isin(training_seasons)
    )

    evaluation_mask = (
        walk_forward_data[
            season_column
        ]
        .eq(evaluation_season)
    )

    walk_forward_fold_records.append(
        {
            "Fold": fold_number,
            "TrainingStartSeason": (
                training_seasons[0]
            ),
            "TrainingEndSeason": (
                training_seasons[-1]
            ),
            "TrainingSeasons": len(
                training_seasons
            ),
            "TrainingFixtures": int(
                training_mask.sum()
            ),
            "EvaluationSeason": (
                evaluation_season
            ),
            "EvaluationFixtures": int(
                evaluation_mask.sum()
            ),
        }
    )


walk_forward_fold_plan = pd.DataFrame(
    walk_forward_fold_records
)


# ------------------------------------------------------------
# Validate the expanding-window structure
# ------------------------------------------------------------

assert (
    walk_forward_fold_plan[
        "EvaluationFixtures"
    ]
    ==
    380
).all(), (
    "Every evaluation fold should contain exactly "
    "380 fixtures."
)

assert (
    walk_forward_fold_plan[
        "TrainingFixtures"
    ]
    ==
    walk_forward_fold_plan[
        "TrainingSeasons"
    ]
    * 380
).all(), (
    "Training fixture counts do not match the number "
    "of complete historical seasons."
)

assert (
    walk_forward_fold_plan[
        "TrainingSeasons"
    ]
    .tolist()
    ==
    [5, 6, 7, 8, 9]
), (
    "The training window does not expand by exactly "
    "one season per fold."
)

assert (
    walk_forward_fold_plan[
        "TrainingFixtures"
    ]
    .tolist()
    ==
    [
        1900,
        2280,
        2660,
        3040,
        3420,
    ]
), (
    "Unexpected training fixture counts were detected."
)


# ------------------------------------------------------------
# Create a compact predictor inventory
# ------------------------------------------------------------

predictor_inventory = pd.DataFrame(
    {
        "Predictor": feature_columns,
        "DataType": [
            str(
                walk_forward_data[
                    column
                ].dtype
            )
            for column in feature_columns
        ],
        "MissingValues": [
            int(
                walk_forward_data[
                    column
                ].isna().sum()
            )
            for column in feature_columns
        ],
        "MissingPercentage": [
            (
                walk_forward_data[
                    column
                ].isna().mean()
                * 100
            )
            for column in feature_columns
        ],
    }
).sort_values(
    by=[
        "MissingPercentage",
        "Predictor",
    ],
    ascending=[
        False,
        True,
    ],
    kind="mergesort",
).reset_index(drop=True)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print(
    "Predictor and walk-forward fold setup "
    "completed successfully."
)

print(
    "Predictors:",
    len(feature_columns),
)

print(
    "Evaluation folds:",
    len(
        walk_forward_fold_plan
    ),
)

print(
    "Out-of-sample fixtures per model:",
    int(
        walk_forward_fold_plan[
            "EvaluationFixtures"
        ].sum()
    ),
)

display(
    walk_forward_fold_plan
)

display(
    predictor_inventory.head(20).round(3)
)

Predictor and walk-forward fold setup completed successfully.
Predictors: 70
Evaluation folds: 5
Out-of-sample fixtures per model: 1900


,Fold,TrainingStartSeason,TrainingEndSeason,TrainingSeasons,TrainingFixtures,EvaluationSeason,EvaluationFixtures
0,1,2015-16,2019-20,5,1900,2020-21,380
1,2,2015-16,2020-21,6,2280,2021-22,380
2,3,2015-16,2021-22,7,2660,2022-23,380
3,4,2015-16,2022-23,8,3040,2023-24,380
4,5,2015-16,2023-24,9,3420,2024-25,380


,Predictor,DataType,MissingValues,MissingPercentage
0,VenueGoalDifferenceFormDifference5,float64,1016,26.737
1,VenueGoalsAgainstFormDifference5,float64,1016,26.737
2,VenueGoalsForFormDifference5,float64,1016,26.737
3,VenuePointsFormDifference5,float64,1016,26.737
4,VenueWinRateFormDifference5,float64,1016,26.737
5,AwayVenueRollingGoalDifference5,float64,1000,26.316
6,AwayVenueRollingGoalsAgainst5,float64,1000,26.316
7,AwayVenueRollingGoalsFor5,float64,1000,26.316
8,AwayVenueRollingPoints5,float64,1000,26.316
9,AwayVenueRollingWinRate5,float64,1000,26.316


### Results and Interpretation

The walk-forward structure was created successfully.

The final modelling dataset contains 70 numeric pre-match predictors and five chronological evaluation folds, covering 1,900 genuinely out-of-sample fixtures per model.

The training sample expands by one complete Premier League season at each fold:

- 1,900 fixtures for the 2020–21 evaluation;
- 2,280 fixtures for 2021–22;
- 2,660 fixtures for 2022–23;
- 3,040 fixtures for 2023–24;
- 3,420 fixtures for 2024–25.

Each evaluation season contains exactly 380 fixtures.

Some rolling and venue-specific predictors contain missing values, particularly where insufficient historical matches were available at the start of a season. These values will be handled using median imputation fitted separately within each training fold.

This preserves the expanding-window design and prevents information from future seasons entering earlier predictions.

## 4. Define Probability Alignment and Evaluation Metrics

The walk-forward backtest compares several models that produce full home-win, draw and away-win probability distributions.

All probabilities must be stored in the project’s standard order:

$$
(H,D,A).
$$

This is important because scikit-learn generally stores class probabilities alphabetically:

$$
(A,D,H).
$$

Using probabilities in the wrong order would produce incorrect log-loss and Brier-score values even when the underlying model predictions were valid.

This section therefore defines helper functions that:

1. reorder classifier probabilities into $(H,D,A)$;
2. validate that every probability row is finite and sums to one;
3. calculate multiclass log loss directly from the observed outcome;
4. calculate the multiclass Brier score;
5. calculate classification accuracy from the highest predicted probability.

Log loss will remain the main model-selection metric because it rewards accurate probabilities and strongly penalises confident incorrect forecasts.

In [3]:
# ============================================================
# 4. Define Probability Alignment and Evaluation Metrics
# ============================================================

from sklearn.metrics import accuracy_score


# ------------------------------------------------------------
# Define the project-wide probability order
# ------------------------------------------------------------

CLASS_ORDER = [
    "H",
    "D",
    "A",
]


# ------------------------------------------------------------
# Validate a probability dataframe
# ------------------------------------------------------------

def validate_probability_frame(
    probability_frame,
    expected_index,
    model_name,
):
    """
    Validate probability order, row alignment and probability sums.
    """

    assert list(
        probability_frame.columns
    ) == CLASS_ORDER, (
        f"{model_name}: probability columns are not "
        "ordered as H, D, A."
    )

    assert probability_frame.index.equals(
        expected_index
    ), (
        f"{model_name}: probability rows are not aligned "
        "with the target rows."
    )

    assert np.isfinite(
        probability_frame.to_numpy()
    ).all(), (
        f"{model_name}: non-finite probabilities were detected."
    )

    assert (
        probability_frame.to_numpy()
        >= 0
    ).all(), (
        f"{model_name}: negative probabilities were detected."
    )

    assert (
        probability_frame.to_numpy()
        <= 1
    ).all(), (
        f"{model_name}: probabilities above one were detected."
    )

    assert np.allclose(
        probability_frame.sum(
            axis=1
        ),
        1.0,
        atol=1e-8,
    ), (
        f"{model_name}: probability rows do not sum to one."
    )


# ------------------------------------------------------------
# Align scikit-learn classifier probabilities
# ------------------------------------------------------------

def align_classifier_probabilities(
    fitted_model,
    predictor_matrix,
    expected_index,
    model_name,
):
    """
    Generate classifier probabilities and reorder them into H, D, A.
    """

    raw_probabilities = (
        fitted_model.predict_proba(
            predictor_matrix
        )
    )

    probability_frame = pd.DataFrame(
        raw_probabilities,
        index=expected_index,
        columns=fitted_model.classes_,
    )

    missing_classes = [
        outcome
        for outcome in CLASS_ORDER
        if outcome not in probability_frame.columns
    ]

    assert not missing_classes, (
        f"{model_name}: missing probability classes "
        f"{missing_classes}."
    )

    probability_frame = (
        probability_frame[
            CLASS_ORDER
        ]
        .copy()
    )

    validate_probability_frame(
        probability_frame,
        expected_index,
        model_name,
    )

    return probability_frame


# ------------------------------------------------------------
# Multiclass log loss
# ------------------------------------------------------------

def multiclass_log_loss(
    y_true,
    probability_frame,
):
    """
    Calculate multiclass log loss using the probability assigned
    to the observed outcome for each fixture.
    """

    assert y_true.index.equals(
        probability_frame.index
    ), (
        "The outcome target and probability rows are not aligned."
    )

    observed_probabilities = np.array(
        [
            probability_frame.loc[
                index,
                observed_outcome,
            ]
            for index, observed_outcome
            in y_true.items()
        ],
        dtype=float,
    )

    observed_probabilities = np.clip(
        observed_probabilities,
        1e-15,
        1.0,
    )

    log_loss_value = -np.mean(
        np.log(
            observed_probabilities
        )
    )

    return float(
        log_loss_value
    )


# ------------------------------------------------------------
# Multiclass Brier score
# ------------------------------------------------------------

def multiclass_brier_score(
    y_true,
    probability_frame,
):
    """
    Calculate the mean squared distance between predicted
    probabilities and the one-hot encoded observed outcomes.
    """

    assert y_true.index.equals(
        probability_frame.index
    ), (
        "The outcome target and probability rows are not aligned."
    )

    observed_outcome_matrix = pd.DataFrame(
        0.0,
        index=y_true.index,
        columns=CLASS_ORDER,
    )

    for outcome in CLASS_ORDER:

        observed_outcome_matrix.loc[
            y_true.eq(
                outcome
            ),
            outcome,
        ] = 1.0

    squared_errors = (
        probability_frame.to_numpy()
        -
        observed_outcome_matrix.to_numpy()
    ) ** 2

    brier_score_value = np.mean(
        np.sum(
            squared_errors,
            axis=1,
        )
    )

    return float(
        brier_score_value
    )


# ------------------------------------------------------------
# Evaluate one model on one walk-forward fold
# ------------------------------------------------------------

def evaluate_probability_model(
    evaluation_season,
    model_name,
    y_true,
    probability_frame,
):
    """
    Return the principal probability metrics for one model
    on one unseen evaluation season.
    """

    validate_probability_frame(
        probability_frame,
        y_true.index,
        model_name,
    )

    predicted_outcomes = (
        probability_frame.idxmax(
            axis=1
        )
    )

    return {
        "EvaluationSeason": (
            evaluation_season
        ),
        "Model": model_name,
        "Fixtures": len(
            y_true
        ),
        "LogLoss": multiclass_log_loss(
            y_true,
            probability_frame,
        ),
        "BrierScore": multiclass_brier_score(
            y_true,
            probability_frame,
        ),
        "Accuracy": accuracy_score(
            y_true,
            predicted_outcomes,
        ),
    }


print(
    "Probability alignment and evaluation helpers "
    "defined successfully."
)

print(
    "Project probability order:",
    CLASS_ORDER,
)

Probability alignment and evaluation helpers defined successfully.
Project probability order: ['H', 'D', 'A']


## 5. Define the Frozen Model Specifications

The walk-forward backtest will compare five previously selected modelling approaches:

1. tuned multinomial logistic regression;
2. tuned Random Forest;
3. tuned Histogram Gradient Boosting;
4. the validation-selected direct probability ensemble;
5. Independent Poisson scoreline modelling.

The hyperparameters selected in the earlier notebooks will remain fixed throughout every walk-forward fold.

This is important because changing the settings after observing each evaluation season would introduce retrospective optimisation.

The frozen specifications are:

### Logistic Regression

- regularisation parameter: `C = 0.01`;
- class weighting: `None`;
- scaled predictors.

### Random Forest

- number of trees: `500`;
- maximum tree depth: `5`;
- minimum observations per leaf: `30`;
- proportion of predictors considered at each split: `0.5`;
- unscaled, median-imputed predictors.

### Histogram Gradient Boosting

- learning rate: `0.05`;
- maximum boosting iterations: `100`;
- maximum leaf nodes: `3`;
- minimum observations per leaf: `30`;
- L2 regularisation: `1.0`;
- unscaled, median-imputed predictors.

### Direct Probability Ensemble

The frozen ensemble combines the three classifier probability distributions using:

$$
P_{\text{ensemble}}
=
0.55P_{\text{logistic}}
+
0.15P_{\text{RF}}
+
0.30P_{\text{HGB}}.
$$

### Independent Poisson

Separate Poisson regressions will model home and away goals using:

$$
\alpha_H = 0,
\qquad
\alpha_A = 0.001.
$$

Both Poisson models will use scaled predictors.

Every model will be fitted from scratch using only the historical training data available within each fold.

In [4]:
# ============================================================
# 5. Define the Frozen Model Specifications
# ============================================================

from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import (
    LogisticRegression,
    PoissonRegressor,
)

from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# Reproducibility settings
# ------------------------------------------------------------

RANDOM_STATE = 42


# ------------------------------------------------------------
# Frozen direct-model ensemble weights
# ------------------------------------------------------------

ENSEMBLE_WEIGHTS = {
    "LogisticRegression": 0.55,
    "RandomForest": 0.15,
    "HistogramGradientBoosting": 0.30,
}


assert np.isclose(
    sum(ENSEMBLE_WEIGHTS.values()),
    1.0,
), (
    "The ensemble weights must sum to one."
)


# ------------------------------------------------------------
# Model-construction helper
# ------------------------------------------------------------

def create_frozen_models():
    """
    Create fresh model instances using the specifications
    selected in the earlier modelling notebooks.

    Fresh model instances are created for every
    walk-forward fold.
    """

    models = {
        "LogisticRegression": LogisticRegression(
            C=0.01,
            class_weight=None,
            solver="lbfgs",
            max_iter=5000,
        ),

        "RandomForest": RandomForestClassifier(
            n_estimators=500,
            max_depth=5,
            min_samples_leaf=30,
            max_features=0.5,
            class_weight=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),

        "HistogramGradientBoosting": (
            HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_iter=100,
                max_leaf_nodes=3,
                min_samples_leaf=30,
                l2_regularization=1.0,
                early_stopping=False,
                random_state=RANDOM_STATE,
            )
        ),

        "HomePoisson": PoissonRegressor(
            alpha=0.001,
            solver="lbfgs",
            max_iter=5000,
            tol=1e-6,
        ),

        "AwayPoisson": PoissonRegressor(
            alpha=0.001,
            solver="lbfgs",
            max_iter=5000,
            tol=1e-6,
        ),
    }

    return models


# ------------------------------------------------------------
# Create and inspect the frozen models
# ------------------------------------------------------------

frozen_models = create_frozen_models()


frozen_model_summary = pd.DataFrame(
    [
        {
            "Model": "Tuned Logistic Regression",
            "PredictorTreatment": (
                "Median imputation and standardisation"
            ),
            "FrozenSpecification": (
                "C=0.01; class_weight=None"
            ),
        },
        {
            "Model": "Tuned Random Forest",
            "PredictorTreatment": (
                "Median imputation only"
            ),
            "FrozenSpecification": (
                "500 trees; max_depth=5; "
                "min_samples_leaf=30; max_features=0.5"
            ),
        },
        {
            "Model": "Tuned Histogram Gradient Boosting",
            "PredictorTreatment": (
                "Median imputation only"
            ),
            "FrozenSpecification": (
                "learning_rate=0.05; max_iter=100; "
                "max_leaf_nodes=3; min_samples_leaf=30; "
                "l2_regularization=1.0"
            ),
        },
        {
            "Model": "Direct Probability Ensemble",
            "PredictorTreatment": (
                "Combination of classifier probabilities"
            ),
            "FrozenSpecification": (
                "55% Logistic; 15% RF; 30% HGB"
            ),
        },
        {
            "Model": "Independent Poisson",
            "PredictorTreatment": (
                "Median imputation and standardisation"
            ),
            "FrozenSpecification": (
                "Home alpha=0.001; Away alpha=0.001"
            ),
        },
    ]
)


# ------------------------------------------------------------
# Validate the generated model objects
# ------------------------------------------------------------

expected_model_keys = {
    "LogisticRegression",
    "RandomForest",
    "HistogramGradientBoosting",
    "HomePoisson",
    "AwayPoisson",
}


assert set(
    frozen_models.keys()
) == expected_model_keys, (
    "The frozen model dictionary is incomplete."
)


assert (
    frozen_models["LogisticRegression"].C
    == 0.01
), (
    "Unexpected logistic-regression specification."
)


assert (
    frozen_models["RandomForest"].n_estimators
    == 500
), (
    "Unexpected Random Forest specification."
)


assert (
    frozen_models[
        "HistogramGradientBoosting"
    ].max_leaf_nodes
    == 3
), (
    "Unexpected HGB specification."
)


assert np.isclose(
    frozen_models["HomePoisson"].alpha,
    0.001,
), (
    "Unexpected home Poisson alpha."
)


assert np.isclose(
    frozen_models["AwayPoisson"].alpha,
    0.001,
), (
    "Unexpected away Poisson alpha."
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print(
    "Frozen model specifications created successfully."
)

print(
    "Ensemble weight total:",
    round(
        sum(ENSEMBLE_WEIGHTS.values()),
        6,
    ),
)

print(
    "Poisson regularisation:",
    {
        "Home": frozen_models[
            "HomePoisson"
        ].alpha,
        "Away": frozen_models[
            "AwayPoisson"
        ].alpha,
    },
)

display(
    frozen_model_summary
)

Frozen model specifications created successfully.
Ensemble weight total: 1.0
Poisson regularisation: {'Home': 0.001, 'Away': 0.001}


,Model,PredictorTreatment,FrozenSpecification
0,Tuned Logistic Regression,Median imputation and standardisation,C=0.01; class_weight=None
1,Tuned Random Forest,Median imputation only,500 trees; max_depth=5; min_samples_leaf=30; m...
2,Tuned Histogram Gradient Boosting,Median imputation only,learning_rate=0.05; max_iter=100; max_leaf_nod...
3,Direct Probability Ensemble,Combination of classifier probabilities,55% Logistic; 15% RF; 30% HGB
4,Independent Poisson,Median imputation and standardisation,Home alpha=0.001; Away alpha=0.001


## 6. Convert Expected Goals into Outcome Probabilities

The Independent Poisson framework predicts separate expected scoring rates for the home and away teams.

For a fixture, these are represented by:

$$
\lambda_H
=
\mathbb{E}[\text{Home Goals}]
$$

and

$$
\lambda_A
=
\mathbb{E}[\text{Away Goals}].
$$

Under a Poisson model, the probability of a team scoring exactly $k$ goals is:

$$
P(X=k)
=
\frac{e^{-\lambda}\lambda^k}{k!}.
$$

Separate home and away goal distributions can therefore be combined into a scoreline probability matrix.

Each matrix cell represents:

$$
P(H=i,A=j)
=
P(H=i)P(A=j),
$$

where $i$ is the number of home goals and $j$ is the number of away goals.

The matrix can then be aggregated into match-outcome probabilities:

$$
P(H)
=
\sum_{i>j}P(H=i,A=j),
$$

$$
P(D)
=
\sum_{i=j}P(H=i,A=j),
$$

and

$$
P(A)
=
\sum_{i<j}P(H=i,A=j).
$$

The scoreline space will be truncated at ten goals for each team. The captured probability mass will then be renormalised so that the final home-win, draw and away-win probabilities sum exactly to one.

In [5]:
# ============================================================
# 6. Convert Expected Goals into Outcome Probabilities
# ============================================================

from scipy.stats import poisson


# ------------------------------------------------------------
# Poisson scoreline settings
# ------------------------------------------------------------

MAX_MODELLED_GOALS = 10

MINIMUM_EXPECTED_GOALS = 1e-6


# ------------------------------------------------------------
# Convert one expected-goal pair into a scoreline matrix
# ------------------------------------------------------------

def create_poisson_scoreline_matrix(
    home_expected_goals,
    away_expected_goals,
    maximum_goals=MAX_MODELLED_GOALS,
):
    """
    Create a renormalised Poisson scoreline probability matrix.

    Rows represent home goals.
    Columns represent away goals.
    """

    home_expected_goals = max(
        float(home_expected_goals),
        MINIMUM_EXPECTED_GOALS,
    )

    away_expected_goals = max(
        float(away_expected_goals),
        MINIMUM_EXPECTED_GOALS,
    )

    goal_values = np.arange(
        maximum_goals + 1
    )

    home_goal_probabilities = poisson.pmf(
        goal_values,
        mu=home_expected_goals,
    )

    away_goal_probabilities = poisson.pmf(
        goal_values,
        mu=away_expected_goals,
    )

    scoreline_matrix = np.outer(
        home_goal_probabilities,
        away_goal_probabilities,
    )

    captured_probability_mass = (
        scoreline_matrix.sum()
    )

    assert captured_probability_mass > 0, (
        "The Poisson scoreline matrix contains no "
        "captured probability mass."
    )

    scoreline_matrix = (
        scoreline_matrix
        /
        captured_probability_mass
    )

    assert np.isfinite(
        scoreline_matrix
    ).all(), (
        "The Poisson scoreline matrix contains "
        "non-finite values."
    )

    assert (
        scoreline_matrix >= 0
    ).all(), (
        "The Poisson scoreline matrix contains "
        "negative probabilities."
    )

    assert np.isclose(
        scoreline_matrix.sum(),
        1.0,
        atol=1e-10,
    ), (
        "The Poisson scoreline matrix does not sum to one."
    )

    return scoreline_matrix


# ------------------------------------------------------------
# Aggregate one scoreline matrix into H/D/A probabilities
# ------------------------------------------------------------

def scoreline_matrix_to_outcome_probabilities(
    scoreline_matrix,
):
    """
    Convert a scoreline matrix into home-win, draw and away-win
    probabilities.
    """

    home_win_probability = np.tril(
        scoreline_matrix,
        k=-1,
    ).sum()

    draw_probability = np.trace(
        scoreline_matrix
    )

    away_win_probability = np.triu(
        scoreline_matrix,
        k=1,
    ).sum()

    outcome_probabilities = np.array(
        [
            home_win_probability,
            draw_probability,
            away_win_probability,
        ],
        dtype=float,
    )

    assert np.isfinite(
        outcome_probabilities
    ).all(), (
        "Non-finite Poisson outcome probabilities were detected."
    )

    assert (
        outcome_probabilities >= 0
    ).all(), (
        "Negative Poisson outcome probabilities were detected."
    )

    assert np.isclose(
        outcome_probabilities.sum(),
        1.0,
        atol=1e-10,
    ), (
        "Poisson H/D/A probabilities do not sum to one."
    )

    return outcome_probabilities


# ------------------------------------------------------------
# Convert several expected-goal predictions into H/D/A
# ------------------------------------------------------------

def expected_goals_to_outcome_probabilities(
    home_expected_goals,
    away_expected_goals,
    expected_index,
):
    """
    Convert arrays of home and away expected goals into a
    probability dataframe ordered as H, D, A.
    """

    home_expected_goals = np.asarray(
        home_expected_goals,
        dtype=float,
    )

    away_expected_goals = np.asarray(
        away_expected_goals,
        dtype=float,
    )

    assert len(
        home_expected_goals
    ) == len(
        away_expected_goals
    ), (
        "Home and away expected-goal arrays have "
        "different lengths."
    )

    assert len(
        home_expected_goals
    ) == len(
        expected_index
    ), (
        "Expected-goal predictions are not aligned "
        "with the fixture index."
    )

    probability_records = []

    for (
        home_lambda,
        away_lambda,
    ) in zip(
        home_expected_goals,
        away_expected_goals,
    ):

        scoreline_matrix = (
            create_poisson_scoreline_matrix(
                home_expected_goals=home_lambda,
                away_expected_goals=away_lambda,
            )
        )

        outcome_probabilities = (
            scoreline_matrix_to_outcome_probabilities(
                scoreline_matrix
            )
        )

        probability_records.append(
            outcome_probabilities
        )

    probability_frame = pd.DataFrame(
        probability_records,
        index=expected_index,
        columns=CLASS_ORDER,
    )

    validate_probability_frame(
        probability_frame=probability_frame,
        expected_index=expected_index,
        model_name="Independent Poisson",
    )

    return probability_frame


# ------------------------------------------------------------
# Test the helper functions using an illustrative fixture
# ------------------------------------------------------------

example_home_expected_goals = 1.60
example_away_expected_goals = 1.10

example_scoreline_matrix = (
    create_poisson_scoreline_matrix(
        home_expected_goals=(
            example_home_expected_goals
        ),
        away_expected_goals=(
            example_away_expected_goals
        ),
    )
)

example_outcome_probabilities = (
    scoreline_matrix_to_outcome_probabilities(
        example_scoreline_matrix
    )
)

example_probability_table = pd.DataFrame(
    [
        {
            "HomeExpectedGoals": (
                example_home_expected_goals
            ),
            "AwayExpectedGoals": (
                example_away_expected_goals
            ),
            "HomeWinProbability": (
                example_outcome_probabilities[0]
            ),
            "DrawProbability": (
                example_outcome_probabilities[1]
            ),
            "AwayWinProbability": (
                example_outcome_probabilities[2]
            ),
            "ProbabilityTotal": (
                example_outcome_probabilities.sum()
            ),
        }
    ]
)

print(
    "Poisson scoreline and outcome-probability "
    "helpers defined successfully."
)

display(
    example_probability_table.round(6)
)

Poisson scoreline and outcome-probability helpers defined successfully.


,HomeExpectedGoals,AwayExpectedGoals,HomeWinProbability,DrawProbability,AwayWinProbability,ProbabilityTotal
0,1.6,1.1,0.489573,0.248911,0.261516,1.0


### Results and Interpretation

The Poisson scoreline conversion functions were validated using an illustrative fixture with expected scoring rates of:

$$
\lambda_H=1.6,
\qquad
\lambda_A=1.1.
$$

The resulting match-outcome probabilities were:

- home win: $48.96\%$;
- draw: $24.89\%$;
- away win: $26.15\%$.

The home team receives the highest win probability because its expected scoring rate exceeds the away team’s expected scoring rate.

The home-win, draw and away-win probabilities sum to exactly one after renormalisation, confirming that the truncated scoreline matrix is being aggregated correctly.

These functions can now be applied to the expected-goal forecasts generated by the Poisson regressions within every walk-forward fold.

## 7. Execute the Expanding-Window Backtest

The complete walk-forward backtest will now be executed.

For each evaluation season:

1. all earlier seasons form the historical training sample;
2. the evaluation season remains completely unseen;
3. median imputation is fitted using the training sample only;
4. standardisation is fitted using the training sample only;
5. fresh versions of the frozen models are created;
6. each model is fitted from scratch;
7. H/D/A probabilities are generated for the unseen season;
8. fixture-level forecasts and season-level evaluation metrics are stored.

The preprocessing differs by model family:

- logistic regression uses median-imputed and standardised predictors;
- Random Forest uses median-imputed predictors without standardisation;
- Histogram Gradient Boosting uses median-imputed predictors without standardisation;
- the direct ensemble combines the three classifier probability distributions;
- the home and away Poisson regressions use median-imputed and standardised predictors.

For the ensemble:

$$
P_{\text{ensemble}}
=
0.55P_{\text{logistic}}
+
0.15P_{\text{RF}}
+
0.30P_{\text{HGB}}.
$$

For the Poisson model, predicted home and away expected goals are converted into scoreline matrices and then aggregated into home-win, draw and away-win probabilities.

The process produces five unseen seasonal evaluations for every model, covering 1,900 out-of-sample fixtures per modelling approach.

In [6]:
# ============================================================
# 7. Execute the Expanding-Window Backtest
# ============================================================

import warnings

from sklearn.exceptions import ConvergenceWarning


# ------------------------------------------------------------
# Create storage objects
# ------------------------------------------------------------

walk_forward_metric_records = []

walk_forward_prediction_frames = []

walk_forward_poisson_goal_frames = []

walk_forward_warning_records = []


# ------------------------------------------------------------
# Loop through each unseen evaluation season
# ------------------------------------------------------------

for fold_number, evaluation_season in enumerate(
    evaluation_seasons,
    start=1,
):

    print(
        f"\nStarting fold {fold_number} of "
        f"{len(evaluation_seasons)}: "
        f"predicting {evaluation_season}"
    )

    # --------------------------------------------------------
    # Identify the expanding historical training window
    # --------------------------------------------------------

    evaluation_position = ordered_seasons.index(
        evaluation_season
    )

    training_seasons = ordered_seasons[
        :evaluation_position
    ]

    training_mask = (
        walk_forward_data[
            season_column
        ]
        .isin(
            training_seasons
        )
    )

    evaluation_mask = (
        walk_forward_data[
            season_column
        ]
        .eq(
            evaluation_season
        )
    )


    # --------------------------------------------------------
    # Create fold-specific predictor matrices
    # --------------------------------------------------------

    X_train = (
        walk_forward_data.loc[
            training_mask,
            feature_columns,
        ]
        .copy()
    )

    X_evaluation = (
        walk_forward_data.loc[
            evaluation_mask,
            feature_columns,
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Create fold-specific outcome targets
    # --------------------------------------------------------

    y_train_outcome = (
        walk_forward_data.loc[
            training_mask,
            target_column,
        ]
        .copy()
    )

    y_evaluation_outcome = (
        walk_forward_data.loc[
            evaluation_mask,
            target_column,
        ]
        .copy()
    )

    y_train_home_goals = (
        walk_forward_data.loc[
            training_mask,
            "HomeGoalsTarget",
        ]
        .astype(int)
        .copy()
    )

    y_train_away_goals = (
        walk_forward_data.loc[
            training_mask,
            "AwayGoalsTarget",
        ]
        .astype(int)
        .copy()
    )

    y_evaluation_home_goals = (
        walk_forward_data.loc[
            evaluation_mask,
            "HomeGoalsTarget",
        ]
        .astype(int)
        .copy()
    )

    y_evaluation_away_goals = (
        walk_forward_data.loc[
            evaluation_mask,
            "AwayGoalsTarget",
        ]
        .astype(int)
        .copy()
    )


    # --------------------------------------------------------
    # Retain fixture identifiers for exported predictions
    # --------------------------------------------------------

    evaluation_metadata = (
        walk_forward_data.loc[
            evaluation_mask,
            fixture_key_columns,
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Validate the fold structure
    # --------------------------------------------------------

    assert len(
        X_train
    ) == len(
        training_seasons
    ) * 380, (
        f"{evaluation_season}: unexpected training size."
    )

    assert len(
        X_evaluation
    ) == 380, (
        f"{evaluation_season}: expected 380 evaluation fixtures."
    )

    assert X_train.index.equals(
        y_train_outcome.index
    ), (
        f"{evaluation_season}: training predictors "
        "and targets are misaligned."
    )

    assert X_evaluation.index.equals(
        y_evaluation_outcome.index
    ), (
        f"{evaluation_season}: evaluation predictors "
        "and targets are misaligned."
    )


    # --------------------------------------------------------
    # Fit fold-specific median imputation
    # --------------------------------------------------------

    fold_imputer = SimpleImputer(
        strategy="median"
    )

    X_train_imputed_array = (
        fold_imputer.fit_transform(
            X_train
        )
    )

    X_evaluation_imputed_array = (
        fold_imputer.transform(
            X_evaluation
        )
    )

    X_train_imputed = pd.DataFrame(
        X_train_imputed_array,
        index=X_train.index,
        columns=feature_columns,
    )

    X_evaluation_imputed = pd.DataFrame(
        X_evaluation_imputed_array,
        index=X_evaluation.index,
        columns=feature_columns,
    )


    # --------------------------------------------------------
    # Fit fold-specific standardisation
    # --------------------------------------------------------

    fold_scaler = StandardScaler()

    X_train_scaled_array = (
        fold_scaler.fit_transform(
            X_train_imputed
        )
    )

    X_evaluation_scaled_array = (
        fold_scaler.transform(
            X_evaluation_imputed
        )
    )

    X_train_scaled = pd.DataFrame(
        X_train_scaled_array,
        index=X_train.index,
        columns=feature_columns,
    )

    X_evaluation_scaled = pd.DataFrame(
        X_evaluation_scaled_array,
        index=X_evaluation.index,
        columns=feature_columns,
    )


    # --------------------------------------------------------
    # Validate fold-specific preprocessing
    # --------------------------------------------------------

    assert np.isfinite(
        X_train_imputed.to_numpy()
    ).all(), (
        f"{evaluation_season}: non-finite values remain "
        "after training imputation."
    )

    assert np.isfinite(
        X_evaluation_imputed.to_numpy()
    ).all(), (
        f"{evaluation_season}: non-finite values remain "
        "after evaluation imputation."
    )

    assert np.isfinite(
        X_train_scaled.to_numpy()
    ).all(), (
        f"{evaluation_season}: non-finite values remain "
        "after training standardisation."
    )

    assert np.isfinite(
        X_evaluation_scaled.to_numpy()
    ).all(), (
        f"{evaluation_season}: non-finite values remain "
        "after evaluation standardisation."
    )


    # --------------------------------------------------------
    # Create fresh frozen models for this fold
    # --------------------------------------------------------

    fold_models = create_frozen_models()


    # --------------------------------------------------------
    # Fit and predict with logistic regression
    # --------------------------------------------------------

    fold_models[
        "LogisticRegression"
    ].fit(
        X_train_scaled,
        y_train_outcome,
    )

    logistic_probabilities = (
        align_classifier_probabilities(
            fitted_model=fold_models[
                "LogisticRegression"
            ],
            predictor_matrix=(
                X_evaluation_scaled
            ),
            expected_index=(
                y_evaluation_outcome.index
            ),
            model_name=(
                "Tuned Logistic Regression"
            ),
        )
    )


    # --------------------------------------------------------
    # Fit and predict with Random Forest
    # --------------------------------------------------------

    fold_models[
        "RandomForest"
    ].fit(
        X_train_imputed,
        y_train_outcome,
    )

    random_forest_probabilities = (
        align_classifier_probabilities(
            fitted_model=fold_models[
                "RandomForest"
            ],
            predictor_matrix=(
                X_evaluation_imputed
            ),
            expected_index=(
                y_evaluation_outcome.index
            ),
            model_name=(
                "Tuned Random Forest"
            ),
        )
    )


    # --------------------------------------------------------
    # Fit and predict with Histogram Gradient Boosting
    # --------------------------------------------------------

    fold_models[
        "HistogramGradientBoosting"
    ].fit(
        X_train_imputed,
        y_train_outcome,
    )

    histogram_gradient_boosting_probabilities = (
        align_classifier_probabilities(
            fitted_model=fold_models[
                "HistogramGradientBoosting"
            ],
            predictor_matrix=(
                X_evaluation_imputed
            ),
            expected_index=(
                y_evaluation_outcome.index
            ),
            model_name=(
                "Tuned Histogram Gradient Boosting"
            ),
        )
    )


    # --------------------------------------------------------
    # Create the frozen direct-model ensemble
    # --------------------------------------------------------

    ensemble_probabilities = (
        ENSEMBLE_WEIGHTS[
            "LogisticRegression"
        ]
        * logistic_probabilities
        +
        ENSEMBLE_WEIGHTS[
            "RandomForest"
        ]
        * random_forest_probabilities
        +
        ENSEMBLE_WEIGHTS[
            "HistogramGradientBoosting"
        ]
        * histogram_gradient_boosting_probabilities
    )

    validate_probability_frame(
        probability_frame=(
            ensemble_probabilities
        ),
        expected_index=(
            y_evaluation_outcome.index
        ),
        model_name=(
            "Direct Probability Ensemble"
        ),
    )


    # --------------------------------------------------------
    # Fit the separate Poisson goal models
    # --------------------------------------------------------

    with warnings.catch_warnings(
        record=True
    ) as captured_warnings:

        warnings.simplefilter(
            "always",
            ConvergenceWarning,
        )

        fold_models[
            "HomePoisson"
        ].fit(
            X_train_scaled,
            y_train_home_goals,
        )

        fold_models[
            "AwayPoisson"
        ].fit(
            X_train_scaled,
            y_train_away_goals,
        )


    # --------------------------------------------------------
    # Store any Poisson convergence warnings
    # --------------------------------------------------------

    for captured_warning in captured_warnings:

        if issubclass(
            captured_warning.category,
            ConvergenceWarning,
        ):

            walk_forward_warning_records.append(
                {
                    "Fold": fold_number,
                    "EvaluationSeason": (
                        evaluation_season
                    ),
                    "WarningCategory": (
                        captured_warning
                        .category
                        .__name__
                    ),
                    "WarningMessage": str(
                        captured_warning.message
                    ),
                }
            )


    # --------------------------------------------------------
    # Predict expected home and away goals
    # --------------------------------------------------------

    evaluation_home_expected_goals = (
        fold_models[
            "HomePoisson"
        ]
        .predict(
            X_evaluation_scaled
        )
    )

    evaluation_away_expected_goals = (
        fold_models[
            "AwayPoisson"
        ]
        .predict(
            X_evaluation_scaled
        )
    )

    evaluation_home_expected_goals = np.clip(
        evaluation_home_expected_goals,
        MINIMUM_EXPECTED_GOALS,
        None,
    )

    evaluation_away_expected_goals = np.clip(
        evaluation_away_expected_goals,
        MINIMUM_EXPECTED_GOALS,
        None,
    )

    assert np.isfinite(
        evaluation_home_expected_goals
    ).all(), (
        f"{evaluation_season}: invalid home expected goals."
    )

    assert np.isfinite(
        evaluation_away_expected_goals
    ).all(), (
        f"{evaluation_season}: invalid away expected goals."
    )


    # --------------------------------------------------------
    # Convert expected goals into H/D/A probabilities
    # --------------------------------------------------------

    poisson_probabilities = (
        expected_goals_to_outcome_probabilities(
            home_expected_goals=(
                evaluation_home_expected_goals
            ),
            away_expected_goals=(
                evaluation_away_expected_goals
            ),
            expected_index=(
                y_evaluation_outcome.index
            ),
        )
    )


    # --------------------------------------------------------
    # Collect the five model probability frames
    # --------------------------------------------------------

    fold_probability_frames = {
        "Tuned Logistic Regression": (
            logistic_probabilities
        ),
        "Tuned Random Forest": (
            random_forest_probabilities
        ),
        "Tuned Histogram Gradient Boosting": (
            histogram_gradient_boosting_probabilities
        ),
        "Direct Probability Ensemble": (
            ensemble_probabilities
        ),
        "Independent Poisson": (
            poisson_probabilities
        ),
    }


    # --------------------------------------------------------
    # Evaluate and store each model
    # --------------------------------------------------------

    for (
        model_name,
        probability_frame,
    ) in fold_probability_frames.items():

        model_metrics = (
            evaluate_probability_model(
                evaluation_season=(
                    evaluation_season
                ),
                model_name=model_name,
                y_true=(
                    y_evaluation_outcome
                ),
                probability_frame=(
                    probability_frame
                ),
            )
        )

        walk_forward_metric_records.append(
            model_metrics
        )


        # ----------------------------------------------------
        # Store fixture-level probability forecasts
        # ----------------------------------------------------

        model_fixture_predictions = (
            evaluation_metadata.copy()
        )

        model_fixture_predictions[
            "Fold"
        ] = fold_number

        model_fixture_predictions[
            "EvaluationSeason"
        ] = evaluation_season

        model_fixture_predictions[
            "Model"
        ] = model_name

        model_fixture_predictions[
            "ActualOutcome"
        ] = y_evaluation_outcome

        model_fixture_predictions[
            "PredictedOutcome"
        ] = probability_frame.idxmax(
            axis=1
        )

        model_fixture_predictions[
            "Probability_H"
        ] = probability_frame["H"]

        model_fixture_predictions[
            "Probability_D"
        ] = probability_frame["D"]

        model_fixture_predictions[
            "Probability_A"
        ] = probability_frame["A"]

        walk_forward_prediction_frames.append(
            model_fixture_predictions
        )


    # --------------------------------------------------------
    # Store Poisson expected-goal forecasts
    # --------------------------------------------------------

    poisson_goal_predictions = (
        evaluation_metadata.copy()
    )

    poisson_goal_predictions[
        "Fold"
    ] = fold_number

    poisson_goal_predictions[
        "EvaluationSeason"
    ] = evaluation_season

    poisson_goal_predictions[
        "ActualHomeGoals"
    ] = y_evaluation_home_goals

    poisson_goal_predictions[
        "ActualAwayGoals"
    ] = y_evaluation_away_goals

    poisson_goal_predictions[
        "ExpectedHomeGoals"
    ] = evaluation_home_expected_goals

    poisson_goal_predictions[
        "ExpectedAwayGoals"
    ] = evaluation_away_expected_goals

    walk_forward_poisson_goal_frames.append(
        poisson_goal_predictions
    )


    # --------------------------------------------------------
    # Print a compact fold completion message
    # --------------------------------------------------------

    fold_metric_preview = pd.DataFrame(
        walk_forward_metric_records
    )

    fold_metric_preview = (
        fold_metric_preview.loc[
            fold_metric_preview[
                "EvaluationSeason"
            ]
            ==
            evaluation_season
        ]
        .sort_values(
            by="LogLoss",
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    print(
        f"Completed {evaluation_season}. "
        f"Best fold model: "
        f"{fold_metric_preview.loc[0, 'Model']} "
        f"with log loss "
        f"{fold_metric_preview.loc[0, 'LogLoss']:.6f}"
    )


# ------------------------------------------------------------
# Combine all stored results
# ------------------------------------------------------------

walk_forward_results = pd.DataFrame(
    walk_forward_metric_records
)

walk_forward_predictions = pd.concat(
    walk_forward_prediction_frames,
    ignore_index=True,
)

walk_forward_poisson_goals = pd.concat(
    walk_forward_poisson_goal_frames,
    ignore_index=True,
)

walk_forward_warnings = pd.DataFrame(
    walk_forward_warning_records
)


# ------------------------------------------------------------
# Validate the completed walk-forward outputs
# ------------------------------------------------------------

assert len(
    walk_forward_results
) == 25, (
    "Expected 25 fold-level model results."
)

assert len(
    walk_forward_predictions
) == (
    5
    * 5
    * 380
), (
    "Unexpected number of fixture-level model predictions."
)

assert len(
    walk_forward_poisson_goals
) == (
    5
    * 380
), (
    "Unexpected number of Poisson goal forecasts."
)

assert (
    walk_forward_results[
        "Fixtures"
    ]
    ==
    380
).all(), (
    "Every model-fold result should contain 380 fixtures."
)

assert walk_forward_predictions[
    [
        "Probability_H",
        "Probability_D",
        "Probability_A",
    ]
].notna().all().all(), (
    "At least one stored probability is missing."
)

assert np.allclose(
    walk_forward_predictions[
        [
            "Probability_H",
            "Probability_D",
            "Probability_A",
        ]
    ].sum(
        axis=1
    ),
    1.0,
    atol=1e-8,
), (
    "At least one stored probability row does not sum to one."
)


# ------------------------------------------------------------
# Display the completed fold-level results
# ------------------------------------------------------------

print(
    "\nWalk-forward backtest completed successfully."
)

print(
    "Fold-level model results:",
    len(
        walk_forward_results
    ),
)

print(
    "Fixture-level probability forecasts:",
    f"{len(walk_forward_predictions):,}",
)

print(
    "Poisson expected-goal forecasts:",
    f"{len(walk_forward_poisson_goals):,}",
)

print(
    "Poisson convergence warnings:",
    len(
        walk_forward_warnings
    ),
)

display(
    walk_forward_results
    .sort_values(
        by=[
            "EvaluationSeason",
            "LogLoss",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
    .round(6)
)

if not walk_forward_warnings.empty:

    display(
        walk_forward_warnings
    )


Starting fold 1 of 5: predicting 2020-21
Completed 2020-21. Best fold model: Tuned Random Forest with log loss 1.056991

Starting fold 2 of 5: predicting 2021-22
Completed 2021-22. Best fold model: Independent Poisson with log loss 0.965925

Starting fold 3 of 5: predicting 2022-23
Completed 2022-23. Best fold model: Direct Probability Ensemble with log loss 0.974600

Starting fold 4 of 5: predicting 2023-24
Completed 2023-24. Best fold model: Independent Poisson with log loss 0.924724

Starting fold 5 of 5: predicting 2024-25
Completed 2024-25. Best fold model: Independent Poisson with log loss 0.989193

Walk-forward backtest completed successfully.
Fold-level model results: 25
Fixture-level probability forecasts: 9,500
Poisson expected-goal forecasts: 1,900
Poisson convergence warnings: 0


,EvaluationSeason,Model,Fixtures,LogLoss,BrierScore,Accuracy
0,2020-21,Tuned Random Forest,380,1.056991,0.626266,0.502632
1,2020-21,Direct Probability Ensemble,380,1.064178,0.629232,0.505263
2,2020-21,Independent Poisson,380,1.065491,0.630305,0.494737
3,2020-21,Tuned Logistic Regression,380,1.077027,0.635736,0.473684
4,2020-21,Tuned Histogram Gradient Boosting,380,1.079455,0.634970,0.502632
5,2021-22,Independent Poisson,380,0.965925,0.574944,0.550000
6,2021-22,Tuned Random Forest,380,0.972121,0.576253,0.547368
7,2021-22,Direct Probability Ensemble,380,0.973540,0.578422,0.550000
8,2021-22,Tuned Histogram Gradient Boosting,380,0.973607,0.579845,0.539474
9,2021-22,Tuned Logistic Regression,380,0.983821,0.583866,0.531579


### Results and Interpretation

The expanding-window walk-forward backtest completed successfully across five unseen Premier League seasons.

Each model generated probability forecasts for 1,900 out-of-sample fixtures, producing 9,500 fixture-level model forecasts in total. The Independent Poisson models also generated expected home and away goals for all 1,900 fixtures.

No Poisson convergence warnings remained after applying a small positive regularisation strength to both scoring models:

$$
\alpha_H=0.001,
\qquad
\alpha_A=0.001.
$$

This was necessary because the engineered predictor matrix was rank deficient, with 70 predictors but a matrix rank of 54. Small positive regularisation therefore provided a uniquely identified and numerically stable coefficient solution.

The winning model varied across seasons:

- Tuned Random Forest achieved the lowest log loss in 2020–21;
- Independent Poisson won in 2021–22;
- the Direct Probability Ensemble won in 2022–23;
- Independent Poisson won in 2023–24;
- Independent Poisson won again in 2024–25.

Across all five evaluation seasons, Independent Poisson achieved the lowest mean log loss:

$$
\overline{\operatorname{LogLoss}}_{\text{Poisson}}
=
0.985002.
$$

The Direct Probability Ensemble ranked second with a mean log loss of 0.987957, followed closely by the Tuned Random Forest at 0.988276.

Independent Poisson therefore won three of the five seasons and achieved the strongest average probability performance. However, it did not dominate every fold, showing that the relative performance of each modelling approach changes across different league environments.

The 2020–21 season was the most difficult evaluation period for every model. This is consistent with the unusual conditions of that season, particularly the disruption to conventional home advantage.

Overall, the results support Independent Poisson as the strongest model across the walk-forward period while also showing that the ensemble and Random Forest remain competitive alternatives.

## 8. Season-by-Season and Walk-Forward Model Ranking

The walk-forward backtest produced one set of performance metrics for each model in each unseen evaluation season.

This section reorganises those results to compare:

- seasonal log loss;
- seasonal Brier score;
- seasonal accuracy;
- the winning model in each season;
- average performance across all five seasons;
- variation in performance between seasons;
- best- and worst-season results;
- the number of seasons won by each model.

Log loss remains the principal ranking metric.

A model with the lowest average log loss provides the strongest probability forecasts across the complete walk-forward period. However, average performance should be considered alongside stability.

A model may achieve a strong average because of one exceptional season while performing inconsistently elsewhere. Seasonal standard deviation and worst-season log loss therefore provide additional evidence about robustness.

In [7]:
# ============================================================
# 8. Season-by-Season and Aggregate Model Comparison
# ============================================================


# ------------------------------------------------------------
# Create season-by-model metric tables
# ------------------------------------------------------------

log_loss_by_season = (
    walk_forward_results
    .pivot(
        index="EvaluationSeason",
        columns="Model",
        values="LogLoss",
    )
    .reindex(
        evaluation_seasons
    )
)

brier_score_by_season = (
    walk_forward_results
    .pivot(
        index="EvaluationSeason",
        columns="Model",
        values="BrierScore",
    )
    .reindex(
        evaluation_seasons
    )
)

accuracy_by_season = (
    walk_forward_results
    .pivot(
        index="EvaluationSeason",
        columns="Model",
        values="Accuracy",
    )
    .reindex(
        evaluation_seasons
    )
)


# ------------------------------------------------------------
# Identify the lowest-log-loss model in each season
# ------------------------------------------------------------

season_winners = (
    walk_forward_results
    .sort_values(
        by=[
            "EvaluationSeason",
            "LogLoss",
        ],
        ascending=[
            True,
            True,
        ],
        kind="mergesort",
    )
    .groupby(
        "EvaluationSeason",
        as_index=False,
        sort=False,
    )
    .first()
    [
        [
            "EvaluationSeason",
            "Model",
            "LogLoss",
            "BrierScore",
            "Accuracy",
        ]
    ]
    .rename(
        columns={
            "Model": "WinningModel",
            "LogLoss": "WinningLogLoss",
            "BrierScore": "WinningBrierScore",
            "Accuracy": "WinningAccuracy",
        }
    )
)

season_winners["EvaluationSeason"] = pd.Categorical(
    season_winners["EvaluationSeason"],
    categories=evaluation_seasons,
    ordered=True,
)

season_winners = (
    season_winners
    .sort_values(
        by="EvaluationSeason"
    )
    .reset_index(drop=True)
)

season_winners["EvaluationSeason"] = (
    season_winners[
        "EvaluationSeason"
    ].astype(str)
)


# ------------------------------------------------------------
# Count the number of seasonal wins achieved by each model
# ------------------------------------------------------------

season_win_counts = (
    season_winners[
        "WinningModel"
    ]
    .value_counts()
    .rename(
        "SeasonsWon"
    )
)


# ------------------------------------------------------------
# Calculate aggregate walk-forward performance
# ------------------------------------------------------------

aggregate_model_comparison = (
    walk_forward_results
    .groupby(
        "Model",
        as_index=False,
    )
    .agg(
        MeanLogLoss=(
            "LogLoss",
            "mean",
        ),
        MedianLogLoss=(
            "LogLoss",
            "median",
        ),
        LogLossStandardDeviation=(
            "LogLoss",
            "std",
        ),
        BestSeasonLogLoss=(
            "LogLoss",
            "min",
        ),
        WorstSeasonLogLoss=(
            "LogLoss",
            "max",
        ),
        MeanBrierScore=(
            "BrierScore",
            "mean",
        ),
        MeanAccuracy=(
            "Accuracy",
            "mean",
        ),
    )
)


aggregate_model_comparison[
    "SeasonsWon"
] = (
    aggregate_model_comparison[
        "Model"
    ]
    .map(
        season_win_counts
    )
    .fillna(0)
    .astype(int)
)


aggregate_model_comparison[
    "LogLossRange"
] = (
    aggregate_model_comparison[
        "WorstSeasonLogLoss"
    ]
    -
    aggregate_model_comparison[
        "BestSeasonLogLoss"
    ]
)


# ------------------------------------------------------------
# Rank each model using mean walk-forward log loss
# ------------------------------------------------------------

aggregate_model_comparison = (
    aggregate_model_comparison
    .sort_values(
        by=[
            "MeanLogLoss",
            "LogLossStandardDeviation",
        ],
        ascending=[
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

aggregate_model_comparison.insert(
    0,
    "WalkForwardRank",
    np.arange(
        1,
        len(
            aggregate_model_comparison
        )
        + 1,
    ),
)


# ------------------------------------------------------------
# Extract headline results
# ------------------------------------------------------------

leading_model = (
    aggregate_model_comparison
    .iloc[0]
)

most_stable_model = (
    aggregate_model_comparison
    .sort_values(
        by=[
            "LogLossStandardDeviation",
            "MeanLogLoss",
        ],
        ascending=[
            True,
            True,
        ],
        kind="mergesort",
    )
    .iloc[0]
)


# ------------------------------------------------------------
# Validate the comparison tables
# ------------------------------------------------------------

assert log_loss_by_season.shape == (
    5,
    5,
), (
    "The seasonal log-loss table should contain "
    "five seasons and five models."
)

assert len(
    season_winners
) == 5, (
    "Exactly one winning model should be recorded "
    "for each evaluation season."
)

assert (
    aggregate_model_comparison[
        "SeasonsWon"
    ].sum()
    ==
    5
), (
    "The number of seasonal wins should sum to five."
)

assert (
    aggregate_model_comparison[
        "WalkForwardRank"
    ].tolist()
    ==
    [
        1,
        2,
        3,
        4,
        5,
    ]
), (
    "The aggregate model ranks are invalid."
)


# ------------------------------------------------------------
# Display the results
# ------------------------------------------------------------

print(
    "Season-by-season and aggregate comparison "
    "completed successfully."
)

print(
    "Leading model by mean log loss:",
    leading_model[
        "Model"
    ],
)

print(
    "Leading mean log loss:",
    round(
        float(
            leading_model[
                "MeanLogLoss"
            ]
        ),
        6,
    ),
)

print(
    "Most stable model:",
    most_stable_model[
        "Model"
    ],
)

print(
    "Lowest seasonal log-loss standard deviation:",
    round(
        float(
            most_stable_model[
                "LogLossStandardDeviation"
            ]
        ),
        6,
    ),
)

print(
    "\nSeasonal log loss:"
)

display(
    log_loss_by_season.round(6)
)

print(
    "Season winners:"
)

display(
    season_winners.round(6)
)

print(
    "Aggregate model comparison:"
)

display(
    aggregate_model_comparison.round(6)
)

Season-by-season and aggregate comparison completed successfully.
Leading model by mean log loss: Independent Poisson
Leading mean log loss: 0.985002
Most stable model: Tuned Random Forest
Lowest seasonal log-loss standard deviation: 0.044268

Seasonal log loss:


Model,Direct Probability Ensemble,Independent Poisson,Tuned Histogram Gradient Boosting,Tuned Logistic Regression,Tuned Random Forest
EvaluationSeason,,,,,
2020-21,1.064178,1.065491,1.079455,1.077027,1.056991
2021-22,0.973540,0.965925,0.973607,0.983821,0.972121
2022-23,0.974600,0.979675,0.985396,0.975053,0.977768
2023-24,0.931185,0.924724,0.937378,0.934551,0.936744
2024-25,0.996282,0.989193,0.996210,1.002164,0.997755


Season winners:


,EvaluationSeason,WinningModel,WinningLogLoss,WinningBrierScore,WinningAccuracy
0,2020-21,Tuned Random Forest,1.056991,0.626266,0.502632
1,2021-22,Independent Poisson,0.965925,0.574944,0.550000
2,2022-23,Direct Probability Ensemble,0.974600,0.580064,0.539474
3,2023-24,Independent Poisson,0.924724,0.545239,0.584211
4,2024-25,Independent Poisson,0.989193,0.592676,0.502632


Aggregate model comparison:


,WalkForwardRank,Model,MeanLogLoss,MedianLogLoss,LogLossStandardDeviation,BestSeasonLogLoss,WorstSeasonLogLoss,MeanBrierScore,MeanAccuracy,SeasonsWon,LogLossRange
0,1,Independent Poisson,0.985002,0.979675,0.051288,0.924724,1.065491,0.585181,0.535789,3,0.140767
1,2,Direct Probability Ensemble,0.987957,0.974600,0.048704,0.931185,1.064178,0.586320,0.539474,1,0.132993
2,3,Tuned Random Forest,0.988276,0.977768,0.044268,0.936744,1.056991,0.586373,0.538947,1,0.120247
3,4,Tuned Histogram Gradient Boosting,0.994409,0.985396,0.052447,0.937378,1.079455,0.589369,0.540000,0,0.142077
4,5,Tuned Logistic Regression,0.994523,0.983821,0.052334,0.934551,1.077027,0.590034,0.527895,0,0.142476


## 9. Statistical Comparison of Out-of-Sample Forecast Losses

The aggregate rankings show that Independent Poisson achieved the lowest mean walk-forward log loss. However, a lower sample mean does not automatically establish that its advantage is statistically distinguishable from ordinary variation.

This section therefore compares Independent Poisson against each direct modelling approach using paired out-of-sample forecast losses.

For fixture $i$, the individual log loss is:

$$
L_i=-\log(p_{i,y_i}),
$$

where $p_{i,y_i}$ is the probability assigned to the outcome that actually occurred.

For comparison model $m$, define the paired loss difference:

$$
d_{i,m}
=
L_{i,m}
-
L_{i,\text{Poisson}}.
$$

Therefore:

- $d_{i,m}>0$ means Independent Poisson produced the lower loss;
- $d_{i,m}<0$ means the comparison model produced the lower loss;
- $d_{i,m}=0$ means the models assigned identical probability to the observed outcome.

Fixtures played on the same date may be affected by common league conditions. The statistical tests will therefore operate on average daily losses rather than treating every fixture as completely independent.

Three forms of evidence will be reported:

1. a paired t-test of average daily losses;
2. a Wilcoxon signed-rank test of paired daily losses;
3. a stratified cluster bootstrap confidence interval obtained by resampling match dates separately within each season.

Because four models are being compared against Poisson, Holm-adjusted p-values will also be calculated to control for multiple comparisons.

Statistical significance will be interpreted alongside the size and consistency of the improvement. A very small statistically detectable difference may still have limited practical importance.

In [8]:
# ============================================================
# 9. Statistical Comparison of Out-of-Sample Forecast Losses
# ============================================================

from scipy.stats import (
    ttest_rel,
    wilcoxon,
)


# ------------------------------------------------------------
# Statistical settings
# ------------------------------------------------------------

BOOTSTRAP_REPETITIONS = 10_000
STATISTICAL_RANDOM_STATE = 42

POISSON_MODEL_NAME = (
    "Independent Poisson"
)

COMPARISON_MODELS = [
    "Tuned Logistic Regression",
    "Tuned Random Forest",
    "Tuned Histogram Gradient Boosting",
    "Direct Probability Ensemble",
]


# ------------------------------------------------------------
# Calculate the loss assigned to every observed outcome
# ------------------------------------------------------------

prediction_loss_data = (
    walk_forward_predictions
    .copy()
)


required_prediction_columns = (
    fixture_key_columns
    + [
        "EvaluationSeason",
        "Model",
        "ActualOutcome",
        "Probability_H",
        "Probability_D",
        "Probability_A",
    ]
)


missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in prediction_loss_data.columns
]


assert not missing_prediction_columns, (
    "The walk-forward prediction table is missing: "
    f"{missing_prediction_columns}"
)


prediction_loss_data[
    "ObservedOutcomeProbability"
] = np.select(
    condlist=[
        prediction_loss_data[
            "ActualOutcome"
        ].eq("H"),

        prediction_loss_data[
            "ActualOutcome"
        ].eq("D"),

        prediction_loss_data[
            "ActualOutcome"
        ].eq("A"),
    ],
    choicelist=[
        prediction_loss_data[
            "Probability_H"
        ],

        prediction_loss_data[
            "Probability_D"
        ],

        prediction_loss_data[
            "Probability_A"
        ],
    ],
    default=np.nan,
)


assert prediction_loss_data[
    "ObservedOutcomeProbability"
].notna().all(), (
    "At least one observed outcome could not be matched "
    "to its predicted probability."
)


assert (
    prediction_loss_data[
        "ObservedOutcomeProbability"
    ]
    > 0
).all(), (
    "Observed-outcome probabilities must be positive."
)


assert (
    prediction_loss_data[
        "ObservedOutcomeProbability"
    ]
    <= 1
).all(), (
    "Observed-outcome probabilities cannot exceed one."
)


prediction_loss_data[
    "FixtureLogLoss"
] = -np.log(
    prediction_loss_data[
        "ObservedOutcomeProbability"
    ]
)


# ------------------------------------------------------------
# Extract the Independent Poisson fixture losses
# ------------------------------------------------------------

pairing_columns = (
    fixture_key_columns
    + [
        "EvaluationSeason",
        "ActualOutcome",
    ]
)


poisson_fixture_losses = (
    prediction_loss_data.loc[
        prediction_loss_data[
            "Model"
        ].eq(
            POISSON_MODEL_NAME
        ),
        pairing_columns
        + [
            "FixtureLogLoss",
        ],
    ]
    .rename(
        columns={
            "FixtureLogLoss": (
                "PoissonLogLoss"
            ),
        }
    )
    .copy()
)


assert len(
    poisson_fixture_losses
) == 1900, (
    "Expected 1,900 Independent Poisson fixture losses."
)


assert not poisson_fixture_losses.duplicated(
    subset=pairing_columns
).any(), (
    "Duplicate Independent Poisson fixture losses "
    "were detected."
)


# ------------------------------------------------------------
# Holm multiple-testing correction
# ------------------------------------------------------------

def holm_adjust_p_values(
    p_values,
):
    """
    Apply the Holm step-down correction to a sequence
    of p-values.
    """

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    number_of_tests = len(
        p_values
    )

    sorted_positions = np.argsort(
        p_values
    )

    adjusted_p_values = np.empty(
        number_of_tests,
        dtype=float,
    )

    previous_adjusted_value = 0.0

    for sorted_rank, original_position in enumerate(
        sorted_positions
    ):

        remaining_tests = (
            number_of_tests
            -
            sorted_rank
        )

        adjusted_value = (
            remaining_tests
            *
            p_values[
                original_position
            ]
        )

        adjusted_value = max(
            adjusted_value,
            previous_adjusted_value,
        )

        adjusted_value = min(
            adjusted_value,
            1.0,
        )

        adjusted_p_values[
            original_position
        ] = adjusted_value

        previous_adjusted_value = (
            adjusted_value
        )

    return adjusted_p_values


# ------------------------------------------------------------
# Compare every direct model against Independent Poisson
# ------------------------------------------------------------

pairwise_statistical_records = []
pairwise_fixture_frames = []

random_generator = np.random.default_rng(
    STATISTICAL_RANDOM_STATE
)


for comparison_model_name in COMPARISON_MODELS:

    comparison_fixture_losses = (
        prediction_loss_data.loc[
            prediction_loss_data[
                "Model"
            ].eq(
                comparison_model_name
            ),
            pairing_columns
            + [
                "FixtureLogLoss",
            ],
        ]
        .rename(
            columns={
                "FixtureLogLoss": (
                    "ComparisonLogLoss"
                ),
            }
        )
        .copy()
    )

    assert len(
        comparison_fixture_losses
    ) == 1900, (
        f"{comparison_model_name}: expected "
        "1,900 fixture losses."
    )

    paired_fixture_losses = (
        comparison_fixture_losses
        .merge(
            poisson_fixture_losses,
            on=pairing_columns,
            how="inner",
            validate="one_to_one",
        )
    )

    assert len(
        paired_fixture_losses
    ) == 1900, (
        f"{comparison_model_name}: the paired loss "
        "table should contain 1,900 fixtures."
    )

    paired_fixture_losses[
        "ComparisonModel"
    ] = comparison_model_name

    paired_fixture_losses[
        "LossDifference"
    ] = (
        paired_fixture_losses[
            "ComparisonLogLoss"
        ]
        -
        paired_fixture_losses[
            "PoissonLogLoss"
        ]
    )

    pairwise_fixture_frames.append(
        paired_fixture_losses
    )

    # --------------------------------------------------------
    # Aggregate fixtures played on the same date
    # --------------------------------------------------------

    daily_paired_losses = (
        paired_fixture_losses
        .groupby(
            [
                "EvaluationSeason",
                date_column,
            ],
            as_index=False,
        )
        .agg(
            ComparisonMeanLogLoss=(
                "ComparisonLogLoss",
                "mean",
            ),
            PoissonMeanLogLoss=(
                "PoissonLogLoss",
                "mean",
            ),
            FixturesOnDate=(
                "LossDifference",
                "size",
            ),
        )
    )

    daily_paired_losses[
        "MeanLossDifference"
    ] = (
        daily_paired_losses[
            "ComparisonMeanLogLoss"
        ]
        -
        daily_paired_losses[
            "PoissonMeanLogLoss"
        ]
    )

    # --------------------------------------------------------
    # Paired t-test and Wilcoxon signed-rank test
    # --------------------------------------------------------

    paired_t_result = ttest_rel(
        daily_paired_losses[
            "ComparisonMeanLogLoss"
        ],
        daily_paired_losses[
            "PoissonMeanLogLoss"
        ],
    )

    wilcoxon_result = wilcoxon(
        daily_paired_losses[
            "ComparisonMeanLogLoss"
        ],
        daily_paired_losses[
            "PoissonMeanLogLoss"
        ],
        zero_method="wilcox",
        correction=False,
    )

    # --------------------------------------------------------
    # Stratified date-cluster bootstrap
    # --------------------------------------------------------

    bootstrap_mean_differences = np.empty(
        BOOTSTRAP_REPETITIONS,
        dtype=float,
    )

    seasonal_daily_groups = {
        season: group.reset_index(
            drop=True
        )
        for season, group
        in daily_paired_losses.groupby(
            "EvaluationSeason",
            sort=False,
        )
    }

    for bootstrap_iteration in range(
        BOOTSTRAP_REPETITIONS
    ):

        weighted_difference_total = 0.0
        sampled_fixture_total = 0

        for season in evaluation_seasons:

            season_daily_losses = (
                seasonal_daily_groups[
                    season
                ]
            )

            number_of_dates = len(
                season_daily_losses
            )

            sampled_positions = (
                random_generator.integers(
                    low=0,
                    high=number_of_dates,
                    size=number_of_dates,
                )
            )

            sampled_daily_losses = (
                season_daily_losses.iloc[
                    sampled_positions
                ]
            )

            weighted_difference_total += (
                sampled_daily_losses[
                    "MeanLossDifference"
                ]
                *
                sampled_daily_losses[
                    "FixturesOnDate"
                ]
            ).sum()

            sampled_fixture_total += int(
                sampled_daily_losses[
                    "FixturesOnDate"
                ].sum()
            )

        bootstrap_mean_differences[
            bootstrap_iteration
        ] = (
            weighted_difference_total
            /
            sampled_fixture_total
        )

    bootstrap_lower_bound = float(
        np.quantile(
            bootstrap_mean_differences,
            0.025,
        )
    )

    bootstrap_upper_bound = float(
        np.quantile(
            bootstrap_mean_differences,
            0.975,
        )
    )

    bootstrap_probability_poisson_better = float(
        np.mean(
            bootstrap_mean_differences
            > 0
        )
    )

    # --------------------------------------------------------
    # Season-level consistency
    # --------------------------------------------------------

    seasonal_mean_differences = (
        paired_fixture_losses
        .groupby(
            "EvaluationSeason"
        )
        [
            "LossDifference"
        ]
        .mean()
    )

    mean_comparison_log_loss = float(
        paired_fixture_losses[
            "ComparisonLogLoss"
        ].mean()
    )

    mean_poisson_log_loss = float(
        paired_fixture_losses[
            "PoissonLogLoss"
        ].mean()
    )

    mean_loss_difference = (
        mean_comparison_log_loss
        -
        mean_poisson_log_loss
    )

    pairwise_statistical_records.append(
        {
            "ComparisonModel": (
                comparison_model_name
            ),
            "MeanComparisonLogLoss": (
                mean_comparison_log_loss
            ),
            "MeanPoissonLogLoss": (
                mean_poisson_log_loss
            ),
            "MeanLossDifference": (
                mean_loss_difference
            ),
            "PoissonRelativeImprovementPercent": (
                100
                *
                mean_loss_difference
                /
                mean_comparison_log_loss
            ),
            "DatesCompared": len(
                daily_paired_losses
            ),
            "SeasonsPoissonBetter": int(
                (
                    seasonal_mean_differences
                    > 0
                ).sum()
            ),
            "PairedTStatistic": float(
                paired_t_result.statistic
            ),
            "PairedTPValue": float(
                paired_t_result.pvalue
            ),
            "WilcoxonStatistic": float(
                wilcoxon_result.statistic
            ),
            "WilcoxonPValue": float(
                wilcoxon_result.pvalue
            ),
            "Bootstrap95Lower": (
                bootstrap_lower_bound
            ),
            "Bootstrap95Upper": (
                bootstrap_upper_bound
            ),
            "BootstrapProbabilityPoissonBetter": (
                bootstrap_probability_poisson_better
            ),
        }
    )


# ------------------------------------------------------------
# Combine and adjust the pairwise results
# ------------------------------------------------------------

pairwise_fixture_loss_comparison = pd.concat(
    pairwise_fixture_frames,
    ignore_index=True,
)


statistical_comparison = pd.DataFrame(
    pairwise_statistical_records
)


statistical_comparison[
    "HolmAdjustedTPValue"
] = holm_adjust_p_values(
    statistical_comparison[
        "PairedTPValue"
    ]
)


statistical_comparison[
    "HolmAdjustedWilcoxonPValue"
] = holm_adjust_p_values(
    statistical_comparison[
        "WilcoxonPValue"
    ]
)


statistical_comparison[
    "BootstrapCIExcludesZero"
] = (
    (
        statistical_comparison[
            "Bootstrap95Lower"
        ]
        > 0
    )
    |
    (
        statistical_comparison[
            "Bootstrap95Upper"
        ]
        < 0
    )
)


statistical_comparison[
    "LowerMeanLossModel"
] = np.where(
    statistical_comparison[
        "MeanLossDifference"
    ]
    > 0,
    "Independent Poisson",
    statistical_comparison[
        "ComparisonModel"
    ],
)


statistical_comparison = (
    statistical_comparison
    .sort_values(
        by="MeanLossDifference",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Final validation and output
# ------------------------------------------------------------

assert len(
    statistical_comparison
) == 4, (
    "Expected four pairwise model comparisons."
)


assert len(
    pairwise_fixture_loss_comparison
) == (
    1900
    *
    len(
        COMPARISON_MODELS
    )
), (
    "The combined pairwise fixture table has "
    "an unexpected number of rows."
)


print(
    "Pairwise statistical comparison completed successfully."
)

print(
    "Positive mean differences indicate lower "
    "Independent Poisson log loss."
)

print(
    "Bootstrap repetitions:",
    f"{BOOTSTRAP_REPETITIONS:,}",
)

display(
    statistical_comparison.round(6)
)

Pairwise statistical comparison completed successfully.
Positive mean differences indicate lower Independent Poisson log loss.
Bootstrap repetitions: 10,000


,ComparisonModel,MeanComparisonLogLoss,MeanPoissonLogLoss,MeanLossDifference,PoissonRelativeImprovementPercent,DatesCompared,SeasonsPoissonBetter,PairedTStatistic,PairedTPValue,WilcoxonStatistic,WilcoxonPValue,Bootstrap95Lower,Bootstrap95Upper,BootstrapProbabilityPoissonBetter,HolmAdjustedTPValue,HolmAdjustedWilcoxonPValue,BootstrapCIExcludesZero,LowerMeanLossModel
0,Tuned Logistic Regression,0.994523,0.985002,0.009522,0.957395,604,4,1.692313,0.091103,82226.0,0.033358,0.003283,0.015876,0.9987,0.364412,0.133433,True,Independent Poisson
1,Tuned Histogram Gradient Boosting,0.994409,0.985002,0.009407,0.946036,604,5,1.513093,0.130780,85505.0,0.172728,0.000339,0.018709,0.9787,0.392340,0.518184,True,Independent Poisson
2,Tuned Random Forest,0.988276,0.985002,0.003274,0.331306,604,3,0.178435,0.858441,89122.0,0.602745,-0.004667,0.010913,0.7914,1.000000,0.958946,False,Independent Poisson
3,Direct Probability Ensemble,0.987957,0.985002,0.002955,0.299105,604,3,0.373384,0.708994,88321.0,0.479473,-0.002705,0.008861,0.8472,1.000000,0.958946,False,Independent Poisson


### Results and Interpretation

Independent Poisson achieved a lower mean fixture-level log loss than every direct modelling approach across the 1,900 walk-forward fixtures.

The estimated mean loss differences were:

| Comparison model | Comparison log loss | Poisson log loss | Difference | Relative improvement |
|---|---:|---:|---:|---:|
| Tuned Logistic Regression | 0.994523 | 0.985002 | 0.009522 | 0.96% |
| Tuned Histogram Gradient Boosting | 0.994409 | 0.985002 | 0.009407 | 0.95% |
| Tuned Random Forest | 0.988276 | 0.985002 | 0.003274 | 0.33% |
| Direct Probability Ensemble | 0.987957 | 0.985002 | 0.002955 | 0.30% |

Positive differences indicate that Independent Poisson assigned more probability to the observed outcomes on average.

The stratified date-cluster bootstrap estimated that Poisson outperformed:

- Logistic Regression in 99.87% of bootstrap samples;
- Histogram Gradient Boosting in 97.87% of bootstrap samples;
- Random Forest in 79.14% of bootstrap samples;
- the Direct Probability Ensemble in 84.72% of bootstrap samples.

The 95% bootstrap confidence interval excluded zero for the comparisons with Logistic Regression and Histogram Gradient Boosting. It did not exclude zero for the comparisons with Random Forest or the Direct Probability Ensemble.

However, none of the paired t-tests or Wilcoxon signed-rank tests remained statistically significant at the 5% level after applying the Holm multiple-comparison correction.

The evidence therefore supports three conclusions.

First, Independent Poisson produced the lowest average out-of-sample loss against every comparison model.

Second, the evidence for an advantage over Logistic Regression and Histogram Gradient Boosting was stronger than the evidence against Random Forest and the ensemble.

Third, the differences between the three leading models were small. Independent Poisson's mean log-loss improvement over Random Forest and the ensemble was approximately 0.3%, and the statistical tests could not rule out sampling variation as the cause of these differences.

Independent Poisson should therefore remain the leading model based on its mean walk-forward performance and three seasonal wins. However, it cannot be described as conclusively superior to the Random Forest or Direct Probability Ensemble.

This distinction between model ranking and statistical certainty prevents small numerical differences from being overstated.

## 10. Export Walk-Forward Results

The walk-forward analysis generated genuinely out-of-sample forecasts for five models across 1,900 Premier League fixtures.

These outputs will now be exported so that later notebooks can use the forecasts without refitting the models.

The principal calibration input contains one row for each fixture and model, including:

- fixture identifiers;
- evaluation season;
- observed outcome;
- predicted outcome;
- home-win probability;
- draw probability;
- away-win probability.

Additional exports preserve:

- fold-level model metrics;
- aggregate model rankings;
- seasonal winners;
- Independent Poisson expected-goal forecasts;
- pairwise statistical comparisons.

These files provide a reproducible boundary between walk-forward model evaluation and the next stage of probability calibration.

In [9]:
# ============================================================
# 10. Export Walk-Forward Results
# ============================================================


# ------------------------------------------------------------
# Create the output directory
# ------------------------------------------------------------

walk_forward_output_directory = (
    project_root
    / "outputs"
    / "walk_forward"
)

walk_forward_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Prepare clean export copies
# ------------------------------------------------------------

walk_forward_predictions_export = (
    walk_forward_predictions.copy()
)

walk_forward_results_export = (
    walk_forward_results.copy()
)

walk_forward_poisson_goals_export = (
    walk_forward_poisson_goals.copy()
)

aggregate_model_comparison_export = (
    aggregate_model_comparison.copy()
)

season_winners_export = (
    season_winners.copy()
)

statistical_comparison_export = (
    statistical_comparison.copy()
)

pairwise_fixture_loss_export = (
    pairwise_fixture_loss_comparison.copy()
)


# ------------------------------------------------------------
# Standardise exported date formatting
# ------------------------------------------------------------

for export_dataframe in [
    walk_forward_predictions_export,
    walk_forward_poisson_goals_export,
    pairwise_fixture_loss_export,
]:

    if date_column in export_dataframe.columns:

        export_dataframe[date_column] = pd.to_datetime(
            export_dataframe[date_column],
            errors="raise",
        ).dt.strftime(
            "%Y-%m-%d"
        )


# ------------------------------------------------------------
# Define export paths
# ------------------------------------------------------------

fixture_probability_path = (
    walk_forward_output_directory
    / "walk_forward_fixture_probabilities.csv"
)

fold_metric_path = (
    walk_forward_output_directory
    / "walk_forward_model_metrics.csv"
)

poisson_expected_goals_path = (
    walk_forward_output_directory
    / "walk_forward_poisson_expected_goals.csv"
)

model_ranking_path = (
    walk_forward_output_directory
    / "walk_forward_model_ranking.csv"
)

season_winner_path = (
    walk_forward_output_directory
    / "walk_forward_season_winners.csv"
)

statistical_comparison_path = (
    walk_forward_output_directory
    / "walk_forward_statistical_comparison.csv"
)

pairwise_fixture_loss_path = (
    walk_forward_output_directory
    / "walk_forward_pairwise_fixture_losses.csv"
)


# ------------------------------------------------------------
# Export every output
# ------------------------------------------------------------

walk_forward_predictions_export.to_csv(
    fixture_probability_path,
    index=False,
)

walk_forward_results_export.to_csv(
    fold_metric_path,
    index=False,
)

walk_forward_poisson_goals_export.to_csv(
    poisson_expected_goals_path,
    index=False,
)

aggregate_model_comparison_export.to_csv(
    model_ranking_path,
    index=False,
)

season_winners_export.to_csv(
    season_winner_path,
    index=False,
)

statistical_comparison_export.to_csv(
    statistical_comparison_path,
    index=False,
)

pairwise_fixture_loss_export.to_csv(
    pairwise_fixture_loss_path,
    index=False,
)


# ------------------------------------------------------------
# Validate the calibration input
# ------------------------------------------------------------

required_calibration_columns = (
    fixture_key_columns
    + [
        "EvaluationSeason",
        "Model",
        "ActualOutcome",
        "PredictedOutcome",
        "Probability_H",
        "Probability_D",
        "Probability_A",
    ]
)

missing_calibration_columns = [
    column
    for column in required_calibration_columns
    if column not in walk_forward_predictions_export.columns
]

assert not missing_calibration_columns, (
    "The calibration export is missing required columns: "
    f"{missing_calibration_columns}"
)

assert len(
    walk_forward_predictions_export
) == 9500, (
    "Expected 9,500 fixture-model probability forecasts."
)

assert (
    walk_forward_predictions_export[
        "Model"
    ].nunique()
    ==
    5
), (
    "Expected forecasts from five models."
)

assert (
    walk_forward_predictions_export[
        "EvaluationSeason"
    ].nunique()
    ==
    5
), (
    "Expected forecasts from five evaluation seasons."
)

assert np.allclose(
    walk_forward_predictions_export[
        [
            "Probability_H",
            "Probability_D",
            "Probability_A",
        ]
    ].sum(
        axis=1
    ),
    1.0,
    atol=1e-8,
), (
    "At least one exported probability row "
    "does not sum to one."
)


# ------------------------------------------------------------
# Confirm every file was created
# ------------------------------------------------------------

export_paths = [
    fixture_probability_path,
    fold_metric_path,
    poisson_expected_goals_path,
    model_ranking_path,
    season_winner_path,
    statistical_comparison_path,
    pairwise_fixture_loss_path,
]

for export_path in export_paths:

    assert export_path.exists(), (
        f"Expected export was not created: {export_path}"
    )


# ------------------------------------------------------------
# Display export summary
# ------------------------------------------------------------

export_summary = pd.DataFrame(
    [
        {
            "Output": "Fixture probabilities",
            "Rows": len(
                walk_forward_predictions_export
            ),
            "File": fixture_probability_path.name,
        },
        {
            "Output": "Fold-level metrics",
            "Rows": len(
                walk_forward_results_export
            ),
            "File": fold_metric_path.name,
        },
        {
            "Output": "Poisson expected goals",
            "Rows": len(
                walk_forward_poisson_goals_export
            ),
            "File": poisson_expected_goals_path.name,
        },
        {
            "Output": "Aggregate model ranking",
            "Rows": len(
                aggregate_model_comparison_export
            ),
            "File": model_ranking_path.name,
        },
        {
            "Output": "Season winners",
            "Rows": len(
                season_winners_export
            ),
            "File": season_winner_path.name,
        },
        {
            "Output": "Statistical comparisons",
            "Rows": len(
                statistical_comparison_export
            ),
            "File": statistical_comparison_path.name,
        },
        {
            "Output": "Pairwise fixture losses",
            "Rows": len(
                pairwise_fixture_loss_export
            ),
            "File": pairwise_fixture_loss_path.name,
        },
    ]
)


print(
    "Walk-forward outputs exported successfully."
)

print(
    "Output directory:",
    walk_forward_output_directory.relative_to(
        project_root
    ),
)

display(
    export_summary
)

Walk-forward outputs exported successfully.
Output directory: outputs\walk_forward


,Output,Rows,File
0,Fixture probabilities,9500,walk_forward_fixture_probabilities.csv
1,Fold-level metrics,25,walk_forward_model_metrics.csv
2,Poisson expected goals,1900,walk_forward_poisson_expected_goals.csv
3,Aggregate model ranking,5,walk_forward_model_ranking.csv
4,Season winners,5,walk_forward_season_winners.csv
5,Statistical comparisons,4,walk_forward_statistical_comparison.csv
6,Pairwise fixture losses,7600,walk_forward_pairwise_fixture_losses.csv


## 11. Conclusions, Limitations and Next Stage

### Final Walk-Forward Conclusion

This notebook evaluated five frozen probability models across five complete unseen Premier League seasons using expanding-window walk-forward backtesting.

Each model produced forecasts for 1,900 out-of-sample fixtures.

The final mean log-loss ranking was:

| Rank | Model | Mean log loss | Seasons won |
|---:|---|---:|---:|
| 1 | Independent Poisson | 0.985002 | 3 |
| 2 | Direct Probability Ensemble | 0.987957 | 1 |
| 3 | Tuned Random Forest | 0.988276 | 1 |
| 4 | Tuned Histogram Gradient Boosting | 0.994409 | 0 |
| 5 | Tuned Logistic Regression | 0.994523 | 0 |

Independent Poisson achieved the lowest mean walk-forward log loss and won three of the five evaluation seasons.

The winning model nevertheless varied through time:

- Random Forest won 2020–21;
- Independent Poisson won 2021–22;
- the Direct Probability Ensemble won 2022–23;
- Independent Poisson won 2023–24;
- Independent Poisson won 2024–25.

This shows that no model dominated every league environment.

### Numerical Stability

The original home-goal Poisson specification used zero regularisation.

A diagnostic analysis showed that the engineered predictor matrix contained 70 predictors but had matrix rank 54. The coefficient system was therefore rank deficient and the unregularised home-goal model did not have a uniquely identified solution.

Both Poisson models were consequently fitted using:

$$
\alpha_H=0.001,
\qquad
\alpha_A=0.001.
$$

This produced a numerically stable backtest with zero convergence warnings.

### Statistical Interpretation

Independent Poisson achieved lower mean fixture-level loss than every comparison model.

Its estimated relative mean log-loss improvements were approximately:

- 0.96% over Logistic Regression;
- 0.95% over Histogram Gradient Boosting;
- 0.33% over Random Forest;
- 0.30% over the Direct Probability Ensemble.

The cluster-bootstrap evidence was strongest against Logistic Regression and Histogram Gradient Boosting.

However, the confidence intervals crossed zero for the comparisons with Random Forest and the Direct Probability Ensemble. None of the paired t-tests or Wilcoxon signed-rank tests remained significant after Holm correction.

Independent Poisson should therefore be treated as the leading model based on average performance and seasonal wins, but not as conclusively superior to the Random Forest or ensemble.

### Strengths

The evaluation framework:

- preserved chronological order;
- fitted preprocessing independently within every fold;
- prevented future seasons from influencing historical forecasts;
- used frozen model specifications;
- evaluated complete probability distributions;
- generated reusable out-of-sample predictions;
- considered both performance magnitude and statistical uncertainty.

### Limitations

Only five complete seasons were available for walk-forward evaluation.

The earliest fold used less training data than later folds, so changes in performance may reflect both changing league conditions and increasing sample size.

The models were refitted at seasonal boundaries rather than before every match.

The frozen hyperparameters were originally selected using a later historical period than some of the early walk-forward folds. The analysis therefore tests the historical robustness of the final specifications rather than implementing a fully nested hyperparameter search within every fold.

The statistical comparisons reduce same-date dependence through daily aggregation and clustered resampling, but football fixtures may still share broader season-level dependencies.

The 2020–21 season was unusually difficult for every model and occurred under exceptional attendance conditions. It remains part of the analysis because excluding an inconvenient season would bias the evaluation.

### Next Stage

The exported fixture-level probability forecasts will be used in:

`07_probability_calibration.ipynb`

The next notebook will test whether predicted probabilities correspond to observed frequencies.

It will investigate:

- home, draw and away reliability curves;
- expected and maximum calibration error;
- model overconfidence and underconfidence;
- expanding-window sigmoid calibration;
- expanding-window isotonic calibration;
- whether calibration improves log loss and Brier score on later unseen seasons.

Calibration models will be trained using earlier walk-forward predictions only and evaluated on later seasons, preserving the chronological structure established in this notebook.